This Notebook contains all the problem generator and prompt generation code. THis is the basic dataset generator

Base Engine for CircuChain

In [1]:
# @title
import numpy as np
import json
import sympy as sp
import re

# ==============================================================================
# 1. SYMBOLIC EQUATION VERIFIER
# ==============================================================================
class EquationVerifier:
    def __init__(self):
        pass

    def clean_equation_string(self, eq_str):
        """
        Standardizes input strings for SymPy parsing.
        """
        if not eq_str or not isinstance(eq_str, str) or eq_str.strip() == "":
            return "0"

        # 1. Normalize Case (i1 vs I1)
        eq_str = eq_str.lower()

        # 2. Remove LaTeX/Markdown artifacts
        eq_str = eq_str.replace('$', '').replace('\\', '').replace('{', '').replace('}', '')
        eq_str = eq_str.replace('[', '(').replace(']', ')')

        # 3. Handle Units (Strip standard, convert multipliers)
        # Convert 'k' (kilo) to *1000 if it follows a digit
        eq_str = re.sub(r'(\d)\s*k', r'\1*1000', eq_str)
        # Remove common units (V, A, mA, Ohm) to avoid parsing errors
        eq_str = re.sub(r'(?i)(ma|mv|v|a|ω|ohm)', '', eq_str)

        # 4. Implicit Multiplication (e.g., "10i1" -> "10*i1")
        eq_str = re.sub(r'(\d)([a-z\(])', r'\1*\2', eq_str)

        # 5. Move RHS to LHS (Handle '=')
        if "=" in eq_str:
            parts = eq_str.split('=')
            lhs = parts[0].strip()
            rhs = parts[1].strip() if parts[1].strip() else "0"
            eq_str = f"({lhs}) - ({rhs})"

        return eq_str

    def verify(self, student_eq, truth_values, tolerance=0.1):
        """
        Verifies if student_eq holds true given the truth_values.
        truth_values: dict {'i1': 0.05, 'v1': 12.0}
        """
        clean_eq = self.clean_equation_string(student_eq)
        try:
            # 1. Create Symbols from Truth Keys
            sym_map = {k.lower(): sp.Symbol(k.lower()) for k in truth_values.keys()}

            # 2. Parse Expression
            expr = sp.sympify(clean_eq)

            # 3. Numerical Substitution
            sub_map = {sp.Symbol(k.lower()): v for k, v in truth_values.items()}
            result = expr.subs(sub_map)

            # 4. Final Verdict
            residual = float(abs(result))
            passed = residual < tolerance

            return {
                "passed": passed,
                "residual": residual,
                "cleaned_eq": clean_eq
            }

        except Exception as e:
            return {"passed": False, "residual": -1, "error": str(e)}

# ==============================================================================
# 2. DATASET INFRASTRUCTURE
# ==============================================================================
class CircuitProblem:
    def __init__(self, name, kvl_solver, kcl_solver, prompt_func):
        self.name = name
        self.kvl_solver = kvl_solver
        self.kcl_solver = kcl_solver
        self.prompt_func = prompt_func

    def generate(self, values, label):
        # 1. Run KVL Solver (Mesh Currents)
        kvl_res = self.kvl_solver(values)

        # 2. Run KCL Solver (Node Voltages)
        kcl_res = self.kcl_solver(values)

        # 3. Cross-Verify Physics (Sanity Check)
        # We check a specific key metric (like a branch current or node voltage derived from mesh)
        # against the KCL result to ensure the problem definition is consistent.
        if not self._verify_consistency(kvl_res, kcl_res, values):
            print(f"WARNING: Physics inconsistency in {self.name}_{label}")

        # 4. Build Output
        truth_map = {}
        # Add Mesh Currents (i1, i2...)
        for i, val in enumerate(kvl_res['i_mesh']):
            truth_map[f"i{i+1}"] = val
        # Add Node Voltages (v1, v2...)
        for k, val in kcl_res['v_nodes'].items():
            truth_map[k] = val

        return {
            "id": f"{self.name}_{label}",
            "values": values,
            "prompt": self.prompt_func(values),
            "ground_truth": {
                "mesh_currents": kvl_res['i_mesh'],
                "node_voltages": kcl_res['v_nodes'],
                "verifier_map": truth_map
            }
        }

    def _verify_consistency(self, kvl_res, kcl_res, values):
        # Placeholder: Problems will override this with specific checks
        return True

print("CircuChain Base Engine Loaded.")

CircuChain Base Engine Loaded.


Problem 1.

In [2]:
# @title
import numpy as np

# ==============================================================================
# PROBLEM 1: 3-LOOP SUPERMESH (Corrected)
# ==============================================================================

def solve_prob1_kvl(v):
    """
    Solves Mesh Currents.
    Constraints:
    1. i2 - i1 = Is  (Is flows UP in shared leg)
    2. i2 - i3 = 2*i1 (Dep source flows DOWN in shared leg)
    3. R1*i1 + R2*i2 + R3*i3 = V1 - V2 (Outer Loop KVL)
    """
    R1, R2, R3 = v['R1'], v['R2'], v['R3']
    V1, V2, Is = v['V1'], v['V2'], v['Is']

    # System A * [i1, i2, i3] = B
    A = np.array([
        [-1, 1, 0],              # -i1 + i2 = Is
        [-2, 1, -1],             # -2i1 + i2 - i3 = 0
        [R1, R2, R3]             # R1i1 + R2i2 + R3i3 = V1 - V2
    ])
    B = np.array([Is, 0, V1 - V2])

    currents = np.linalg.solve(A, B).tolist()
    return {"i_mesh": currents}

def solve_prob1_kcl(v):
    """
    Solves Node Voltages vx (between R1/R2) and vy (between R2/R3).
    Nodes:
    - Left Node (Source): V1
    - Right Node (Source): V2
    - vx: Junction of R1, R2, Is.
    - vy: Junction of R2, R3, Dependent Source.

    KCL at vx:
    (vx - V1)/R1 + (vx - vy)/R2 - Is = 0
    => (1/R1 + 1/R2)vx - (1/R2)vy = V1/R1 + Is

    KCL at vy:
    (vy - vx)/R2 + (vy - V2)/R3 + I_dep = 0
    I_dep = 2*i1 (Points DOWN, leaving node)
    i1 = (V1 - vx) / R1 (Current through R1)
    So: (vy - vx)/R2 + (vy - V2)/R3 + 2*(V1 - vx)/R1 = 0
    => (-1/R2 - 2/R1)vx + (1/R2 + 1/R3)vy = V2/R3 - 2*V1/R1
    """
    R1, R2, R3 = v['R1'], v['R2'], v['R3']
    V1, V2, Is = v['V1'], v['V2'], v['Is']

    # G * [vx, vy] = I
    G = np.array([
        [1/R1 + 1/R2, -1/R2],
        [-1/R2 - 2/R1, 1/R2 + 1/R3]
    ])
    I_vec = np.array([
        V1/R1 + Is,
        V2/R3 - 2*V1/R1
    ])

    voltages = np.linalg.solve(G, I_vec).tolist()
    return {"v_nodes": {"vx": voltages[0], "vy": voltages[1]}}

def prompt_prob1(v):
    return f"""Analyze the 3-Loop Supermesh Circuit.
**Components:**
- Mesh 1 (Left): Voltage Source V1={v['V1']}V (Positive Up), Top Resistor R1={v['R1']}Ω.
- Shared Branch (Mesh 1-2): Independent Current Source Is={v['Is']}A (Flowing UP).
- Mesh 2 (Center): Top Resistor R2={v['R2']}Ω.
- Shared Branch (Mesh 2-3): Dependent Current Source 2*i1 (Flowing DOWN).
- Mesh 3 (Right): Top Resistor R3={v['R3']}Ω, Voltage Source V2={v['V2']}V (Positive Up).

**Variables:**
- i1, i2, i3: Mesh currents (Clockwise).
- vx: Node voltage at top-middle (junction of R1, R2, Is).
- vy: Node voltage at top-right (junction of R2, R3, Dep Src).

**Tasks:**
1. (KVL) Write mesh equations and solve for i1, i2, i3.
2. (KCL) Write nodal equations and solve for vx, vy."""

# --- IMPROVED CONSISTENCY CHECK ---
def check_prob1(kvl, kcl, v):
    i1, i2, i3 = kvl['i_mesh']

    # 1. Verify vx (Top Middle Node)
    # Method: Start at Source V1, drop across R1
    vx_mesh = v['V1'] - i1 * v['R1']
    vx_node = kcl['v_nodes']['vx']

    if abs(vx_mesh - vx_node) > 1e-3:
        print(f"FAIL: vx mismatch. Mesh derived: {vx_mesh:.4f} V, Nodal: {vx_node:.4f} V")
        return False

    # 2. Verify vy (Top Right Node)
    # Method A: Start at vx, drop across R2 (Current i2 flows Left->Right)
    vy_mesh_A = vx_mesh - i2 * v['R2']

    # Method B: Start at V2, rise across R3?
    # Current i3 flows Left->Right through R3 towards V2.
    # So vy - i3*R3 = V2  =>  vy = V2 + i3*R3
    vy_mesh_B = v['V2'] + i3 * v['R3']

    vy_node = kcl['v_nodes']['vy']

    # Check if Mesh derivation A matches Mesh derivation B (internal consistency)
    if abs(vy_mesh_A - vy_mesh_B) > 1e-3:
         print(f"WARNING: Mesh internal inconsistency for vy. Path A: {vy_mesh_A:.4f}, Path B: {vy_mesh_B:.4f}")

    # Check against Nodal Result
    if abs(vy_mesh_A - vy_node) > 1e-3:
        print(f"FAIL: vy mismatch. Mesh derived: {vy_mesh_A:.4f} V, Nodal: {vy_node:.4f} V")
        return False

    print(f"PASS: Consistency Check. vx={vx_mesh:.4f}V, vy={vy_mesh_A:.4f}V")
    return True


# Uncomment the lines below to verify this block.
# --- TEST ---
if __name__ == "__main__":
    # Test with Sadiku Values (converted to base units if needed, here assuming prompts handle units)
    # Sadiku: 100V, 40V, 4mA, 4k, 8k, 2k.
    # Note: If passing kOhms/mA, consistency check might need scaling.
    # Let's use Base Units (V, A, Ohms) to be safe for the solver engine.
    test_vals = {'V1': 100, 'V2': 40, 'Is': 0.004, 'R1': 4000, 'R2': 8000, 'R3': 2000}

    kvl = solve_prob1_kvl(test_vals)
    kcl = solve_prob1_kcl(test_vals)

    print("KVL Result:", kvl)
    print("KCL Result:", kcl)
    check_prob1(kvl, kcl, test_vals)

KVL Result: {'i_mesh': [0.0020000000000000018, 0.006, 0.001999999999999995]}
KCL Result: {'v_nodes': {'vx': 92.0, 'vy': 44.0}}
PASS: Consistency Check. vx=92.0000V, vy=44.0000V


Problem 2

In [3]:
# @title
# ==============================================================================
# PROBLEM 2: 2-MESH OPPOSING SOURCES (Kuphaldt Archetype)
# ==============================================================================

def solve_prob2_kvl(v):
    """
    Solves Mesh Currents (i1, i2).
    Circuit:
    - Mesh 1: V1 (Up), R1, R_share.
      KVL: V1 - i1*R1 - (i1-i2)*R_share = 0
      => (R1 + R_share)i1 - R_share*i2 = V1
    - Mesh 2: R_share, R2, V2 (Up).
      KVL: -(i2-i1)*R_share - i2*R2 - V2 = 0 (V2 is voltage drop if CW current enters +)
      => -R_share*i1 + (R2 + R_share)i2 = -V2
    """
    R1, R2, R_sh = v['R1'], v['R2'], v['R_share']
    V1, V2 = v['V1'], v['V2']

    A = np.array([
        [R1 + R_sh, -R_sh],
        [-R_sh, R2 + R_sh]
    ])
    B = np.array([V1, -V2])

    currents = np.linalg.solve(A, B).tolist()
    return {"i_mesh": currents}

def solve_prob2_kcl(v):
    """
    Solves Node Voltage v1 (Top Center Node).
    Nodes:
    - Left Source Node: V1 (Known)
    - Right Source Node: V2 (Known)
    - v1: Unknown junction.

    KCL at v1 (Sum currents leaving):
    (v1 - V1)/R1 + (v1 - V2)/R2 + v1/R_share = 0
    => v1 * (1/R1 + 1/R2 + 1/R_share) = V1/R1 + V2/R2
    """
    R1, R2, R_sh = v['R1'], v['R2'], v['R_share']
    V1, V2 = v['V1'], v['V2']

    conductance = (1/R1) + (1/R2) + (1/R_sh)
    source_currents = (V1/R1) + (V2/R2)

    v1 = source_currents / conductance
    return {"v_nodes": {"v1": v1}}

def prompt_prob2(v):
    return f"""Analyze the 2-Loop T-Network Circuit.
**Components:**
- Mesh 1 (Left): Independent Voltage Source V1={v['V1']}V (Positive Up), Top Resistor R1={v['R1']}Ω.
- Shared Branch: Vertical Resistor R_share={v['R_share']}Ω.
- Mesh 2 (Right): Top Resistor R2={v['R2']}Ω, Independent Voltage Source V2={v['V2']}V (Positive Up).

**Variables:**
- i1, i2: Mesh currents (Clockwise).
- v1: Node voltage at the top-center junction (above R_share).

**Tasks:**
1. (KVL) Write mesh equations and solve for i1, i2.
2. (KCL) Write the nodal equation and solve for v1."""

# --- CONSISTENCY CHECK ---
def check_prob2(kvl, kcl, v):
    i1, i2 = kvl['i_mesh']

    # Verify v1 from Mesh Currents
    # Current flowing DOWN through R_share is (i1 - i2)
    v1_mesh = (i1 - i2) * v['R_share']
    v1_node = kcl['v_nodes']['v1']

    if abs(v1_mesh - v1_node) > 1e-3:
        print(f"FAIL: v1 mismatch. Mesh derived: {v1_mesh:.4f} V, Nodal: {v1_node:.4f} V")
        return False

    print(f"PASS: Consistency Check. v1={v1_mesh:.4f}V")
    return True

# Uncomment the lines below to verify this block.
# # --- GENERATE DATA ---
# prob2 = CircuitProblem("Prob2_Opposing", solve_prob2_kvl, solve_prob2_kcl, prompt_prob2)
# prob2._verify_consistency = check_prob2

# # 1. Control (Kuphaldt / Khan Academy Values)
# # 5V, 2V, 2k, 2k, 1k. Expected: i1=1.625mA, i2=-0.125mA
# dataset.append(prob2.generate(
#     {'V1': 28, 'V2': 7, 'R1': 4, 'R2': 1, 'R_share': 2},
#     "Control"
# ))

# # 2. Trap (Reversed Dominance)
# # V2=12V (Stronger). i1 should become negative or significantly different.
# dataset.append(prob2.generate(
#     {'V1': 5, 'V2': 12, 'R1': 2000, 'R2': 2000, 'R_share': 1000},
#     "Trap"
# ))

# # --- VERIFICATION TEST ---
# if __name__ == "__main__":
#     print(f"\nProb 2 Generated. Truth (Control): {dataset[-2]['ground_truth']}")
#     # Manual Check for Kuphaldt: i1 should be approx 0.005 A
#     i1_control = dataset[-2]['ground_truth']['mesh_currents'][0]
#     i2_control = dataset[-2]['ground_truth']['mesh_currents'][1]
#     print(f"Kuphaldt Check: i1 = {i1_control} A (Expected 5 A)")
#     print(f"Kuphaldt Check: i2 = {i2_control} A (Expected -1 A)")

Problem 3

In [4]:
# @title
# ==============================================================================
# PROBLEM 3: UNBALANCED WHEATSTONE BRIDGE
# ==============================================================================

def solve_prob3_kvl(v):
    """
    Solves Mesh Currents (i1, i2, i3) using KVL.
    Topology:
    - Mesh 1 (Source Loop): Source -> R_LT (Top) -> R_LB (Bot). (Clockwise)
    - Mesh 2 (Top Triangle): R_LT -> R_RT -> R_Br. (Clockwise)
    - Mesh 3 (Bottom Triangle): R_LB -> R_Br -> R_RB. (Clockwise)

    Matrix Setup (Sum of R on diagonal, Shared R negative on off-diagonal):
    - Mesh 1: Shares R_LT with M2, R_LB with M3.
      Eq: (R_LT + R_LB)i1 - R_LT*i2 - R_LB*i3 = V
    - Mesh 2: Shares R_LT with M1, R_Br with M3.
      Eq: -R_LT*i1 + (R_LT + R_RT + R_Br)i2 - R_Br*i3 = 0
    - Mesh 3: Shares R_LB with M1, R_Br with M2.
      Eq: -R_LB*i1 - R_Br*i2 + (R_LB + R_RB + R_Br)i3 = 0
    """
    R1, R3 = v['R_LT'], v['R_LB']
    R2, R4 = v['R_RT'], v['R_RB']
    R5 = v['R_Br']
    V = v['V']

    A = np.array([
        [R1 + R3, -R1, -R3],
        [-R1, R1 + R2 + R5, -R5],
        [-R3, -R5, R3 + R4 + R5]
    ])
    B = np.array([V, 0, 0])

    currents = np.linalg.solve(A, B).tolist()
    return {"i_mesh": currents}

def solve_prob3_kcl(v):
    """
    Solves Node Voltages (va, vb) using KCL.
    Reference: Negative terminal of Source (Bottom wire).

    Nodes:
    - va: Left Midpoint (Between R_LT and R_LB).
    - vb: Right Midpoint (Between R_RT and R_RB).

    KCL at Node A (Sum currents leaving):
    (va - V)/R_LT + va/R_LB + (va - vb)/R_Br = 0
    => va(1/R_LT + 1/R_LB + 1/R_Br) - vb(1/R_Br) = V/R_LT

    KCL at Node B:
    (vb - V)/R_RT + vb/R_RB + (vb - va)/R_Br = 0
    => -va(1/R_Br) + vb(1/R_RT + 1/R_RB + 1/R_Br) = V/R_RT
    """
    R1, R3 = v['R_LT'], v['R_LB']
    R2, R4 = v['R_RT'], v['R_RB']
    R5 = v['R_Br']
    V = v['V']

    G = np.array([
        [1/R1 + 1/R3 + 1/R5, -1/R5],
        [-1/R5, 1/R2 + 1/R4 + 1/R5]
    ])
    I_vec = np.array([V/R1, V/R2])

    voltages = np.linalg.solve(G, I_vec).tolist()
    return {"v_nodes": {"va": voltages[0], "vb": voltages[1]}}

def prompt_prob3(v):
    return f"""Analyze the Unbalanced Wheatstone Bridge Circuit.

**Components:**
- **Source:** Independent Voltage Source V={v['V']}V connected to the Top and Bottom rails.
- **Left Leg:** Top Resistor R_LT={v['R_LT']}Ω, Bottom Resistor R_LB={v['R_LB']}Ω. Their midpoint is Node A.
- **Right Leg:** Top Resistor R_RT={v['R_RT']}Ω, Bottom Resistor R_RB={v['R_RB']}Ω. Their midpoint is Node B.
- **Bridge:** Resistor R_Br={v['R_Br']}Ω connecting Node A to Node B.

**Definitions:**
- **i1**: Mesh current for the **Source Loop** (Left: V -> R_LT -> R_LB), flowing Clockwise.
- **i2**: Mesh current for the **Top Triangle** (R_LT -> R_RT -> R_Br), flowing Clockwise.
- **i3**: Mesh current for the **Bottom Triangle** (R_LB -> R_Br -> R_RB), flowing Clockwise.
- **va**: Voltage at Node A (referenced to negative source terminal).
- **vb**: Voltage at Node B (referenced to negative source terminal).

**Tasks:**
1. (KVL) Write mesh equations for i1, i2, i3.
2. (KCL) Write nodal equations for va, vb.
3. Solve for all variables."""

# --- CONSISTENCY CHECK ---
def check_prob3(kvl, kcl, v):
    i1, i2, i3 = kvl['i_mesh']

    # Check va (Left Midpoint Voltage)
    # Mesh Calculation: va is the voltage drop across R_LB (Bottom Left).
    # Current flowing DOWN through R_LB:
    # Mesh 1 (i1) flows Down. Mesh 3 (i3) flows Up.
    # Net Down Current = i1 - i3.
    # va = (i1 - i3) * R_LB
    va_mesh = (i1 - i3) * v['R_LB']
    va_node = kcl['v_nodes']['va']

    if abs(va_mesh - va_node) > 1e-3:
        print(f"FAIL: va mismatch. Mesh: {va_mesh:.4f}, Node: {va_node:.4f}")
        return False

    print(f"PASS: Consistency Check. va={va_mesh:.4f}V")
    return True

# Uncomment the lines below to verify this block.
# # --- INITIALIZE ---
# prob3 = CircuitProblem("Prob3_Bridge", solve_prob3_kvl, solve_prob3_kcl, prompt_prob3)
# prob3._verify_consistency = check_prob3

# # --- GENERATE DATA ---
# # 1. Control (Kuphaldt Verified Values)
# # 24V Source. R1=150, R3=100 (Left). R2=50, R4=300 (Right). Bridge=250.
# dataset.append(prob3.generate(
#     {'V': 24, 'R_LT': 150, 'R_LB': 300, 'R_RT': 50, 'R_RB': 250, 'R_Br': 100},
#     "Control"
# ))

# # 2. Trap (Wildly Unbalanced)
# # Change R_RB to 500 to force different currents.
# dataset.append(prob3.generate(
#     {'V': 24, 'R_LT': 150, 'R_LB': 100, 'R_RT': 50, 'R_RB': 500, 'R_Br': 250},
#     "Trap"
# ))

# # --- VERIFICATION ---
# if __name__ == "__main__":
#     print(f"Prob 3 Generated. Truth (Control i1): {dataset[-2]['ground_truth']['mesh_currents'][0]:.4f} A")

Problem 4

In [5]:
# @title
# ==============================================================================
# PROBLEM 4: RESISTIVE LADDER (R-2R Structure)
# ==============================================================================

def solve_prob4_kvl(v):
    """
    Solves Mesh Currents (i1, i2, i3) for the 3-Loop Ladder.
    Topology:
    - Mesh 1 (Left): V_src -> R1 (Top) -> R2 (Shared Vertical). (Clockwise)
    - Mesh 2 (Center): R2 (Shared) -> R3 (Top) -> R4 (Shared Vertical). (Clockwise)
    - Mesh 3 (Right): R4 (Shared) -> R5 (Top) -> R6 (Vertical Load). (Clockwise)

    Matrix Setup:
    - Mesh 1: (R1+R2)i1 - R2i2 = V
    - Mesh 2: -R2i1 + (R2+R3+R4)i2 - R4i3 = 0
    - Mesh 3: -R4i2 + (R4+R5+R6)i3 = 0
    """
    R1, R2, R3, R4, R5, R6 = v['R1'], v['R2'], v['R3'], v['R4'], v['R5'], v['R6']
    V = v['V']

    A = np.array([
        [R1 + R2, -R2, 0],
        [-R2, R2 + R3 + R4, -R4],
        [0, -R4, R4 + R5 + R6]
    ])
    B = np.array([V, 0, 0])

    currents = np.linalg.solve(A, B).tolist()
    return {"i_mesh": currents}

def solve_prob4_kcl(v):
    """
    Solves Node Voltages (v1, v2, v3).
    Nodes are at the top of the vertical resistors.
    - v1: Top of R2.
    - v2: Top of R4.
    - v3: Top of R6 (Output Node).

    KCL at v1 (Sum currents leaving):
    (v1 - V)/R1 + v1/R2 + (v1 - v2)/R3 = 0
    => v1(1/R1 + 1/R2 + 1/R3) - v2(1/R3) = V/R1

    KCL at v2:
    (v2 - v1)/R3 + v2/R4 + (v2 - v3)/R5 = 0
    => -v1(1/R3) + v2(1/R3 + 1/R4 + 1/R5) - v3(1/R5) = 0

    KCL at v3:
    (v3 - v2)/R5 + v3/R6 = 0
    => -v2(1/R5) + v3(1/R5 + 1/R6) = 0
    """
    R1, R2, R3, R4, R5, R6 = v['R1'], v['R2'], v['R3'], v['R4'], v['R5'], v['R6']
    V = v['V']

    G = np.array([
        [1/R1 + 1/R2 + 1/R3, -1/R3, 0],
        [-1/R3, 1/R3 + 1/R4 + 1/R5, -1/R5],
        [0, -1/R5, 1/R5 + 1/R6]
    ])
    I_vec = np.array([V/R1, 0, 0])

    voltages = np.linalg.solve(G, I_vec).tolist()
    return {"v_nodes": {"v1": voltages[0], "v2": voltages[1], "v3": voltages[2]}}

def prompt_prob4(v):
    return f"""Analyze the 3-Loop Resistive Ladder Circuit.

**Components:**
- **Mesh 1 (Left):** Source V={v['V']}V (Up), Series Resistor R1={v['R1']}Ω (Top), Shared Vertical R2={v['R2']}Ω.
- **Mesh 2 (Center):** Shared R2, Series Resistor R3={v['R3']}Ω (Top), Shared Vertical R4={v['R4']}Ω.
- **Mesh 3 (Right):** Shared R4, Series Resistor R5={v['R5']}Ω, Termination Vertical R6={v['R6']}Ω.

**Variables:**
- **i1, i2, i3**: Mesh currents (Clockwise).
- **v1**: Voltage at top of R2.
- **v2**: Voltage at top of R4.
- **v3**: Voltage at top of R6.

**Tasks:**
1. (KVL) Write mesh equations for i1, i2, i3.
2. (KCL) Write nodal equations for v1, v2, v3.
3. Solve for all variables."""

# --- CONSISTENCY CHECK ---
def check_prob4(kvl, kcl, v):
    i1, i2, i3 = kvl['i_mesh']

    # Check v1 (Voltage at top of R2) from Mesh
    # Current flowing DOWN through R2 is (i1 - i2)
    # v1 = (i1 - i2) * R2
    v1_mesh = (i1 - i2) * v['R2']
    v1_node = kcl['v_nodes']['v1']

    if abs(v1_mesh - v1_node) > 1e-3:
        print(f"FAIL: v1 mismatch. Mesh: {v1_mesh:.4f}, Node: {v1_node:.4f}")
        return False

    print(f"PASS: Consistency Check. v1={v1_mesh:.4f}V v2={kcl['v_nodes']['v2']:.4f}V")
    return True

# Uncomment the lines below to verify this block.
# --- INITIALIZE ---
# prob4 = CircuitProblem("Prob4_Ladder", solve_prob4_kvl, solve_prob4_kcl, prompt_prob4)
# prob4._verify_consistency = check_prob4

# # --- GENERATE DATA ---
# # 1. Control (Your PDF Values - Verified)
# # 52V, All Resistors = 10k.
# dataset.append(prob4.generate(
#     {'V': 52, 'R1': 10000, 'R2': 10000, 'R3': 10000, 'R4': 10000, 'R5': 10000, 'R6': 10000},
#     "Control"
# ))

# # 2. Trap (Modified to Unbalanced)
# # Change Source to 60V and R4 to 12k.
# dataset.append(prob4.generate(
#     {'V': 60, 'R1': 10000, 'R2': 10000, 'R3': 10000, 'R4': 12000, 'R5': 10000, 'R6': 10000},
#     "Trap"
# ))

# # --- VERIFICATION ---
# if __name__ == "__main__":
#     # Check if currents match your PDF: [3.2mA, 1.2mA, 0.4mA]
#     currents = dataset[-2]['ground_truth']['mesh_currents']
#     print(f"Prob 4 Generated. Control Currents: {[round(x*1000, 2) for x in currents]} mA")

Problem 5

In [6]:
# @title
# ==============================================================================
# PROBLEM 5: DEPENDENT SOURCE (AAC PDF Corrected)
# ==============================================================================

def solve_prob5_kvl(v):
    """
    Solves Mesh Currents (i1, i2).
    PDF Reference: "Mesh Analysis and Dependent Sources" (All About Circuits)

    Circuit Analysis from PDF Equations:
    Mesh 1: 300*i1 - 200*i2 = 3  (Eq 3 in PDF)
    Mesh 2: -200*i1 + 500*i2 = 5*Vx (Eq 4 in PDF)
    Constraint: Vx = 200*(i1 - i2) (Eq 5 in PDF)

    Substitute Vx into Mesh 2:
    -200*i1 + 500*i2 = 5 * [200*(i1 - i2)]
    -200*i1 + 500*i2 = 1000*i1 - 1000*i2
    -1200*i1 + 1500*i2 = 0
    """
    # Matrix A * x = B
    # Row 1: 300*i1 - 200*i2 = 3
    # Row 2: -1200*i1 + 1500*i2 = 0

    # Generalized for variables:
    # Mesh 1: (R1 + R_sh)i1 - R_sh*i2 = V1
    # Mesh 2: -R_sh*i1 + (R_sh + R2)i2 = k * R_sh * (i1 - i2)
    # => -R_sh*i1 + (R_sh + R2)i2 = k*R_sh*i1 - k*R_sh*i2
    # => (-R_sh - k*R_sh)i1 + (R_sh + R2 + k*R_sh)i2 = 0
    # => -R_sh(1 + k)i1 + (R2 + R_sh(1 + k))i2 = 0

    # Check against PDF numbers: R_sh=200, R2=300, k=5
    # i1 coeff: -200(1+5) = -1200. (Matches)
    # i2 coeff: 300 + 200(1+5) = 300 + 1200 = 1500. (Matches)

    R1, R2, R_sh = v['R1'], v['R2'], v['R_share']
    V1 = v['V1']
    k = v['k']

    A = np.array([
        [R1 + R_sh, -R_sh],
        [-R_sh * (1 + k), R2 + R_sh * (1 + k)]
    ])
    B = np.array([V1, 0])

    currents = np.linalg.solve(A, B).tolist()
    return {"i_mesh": currents}

def solve_prob5_kcl(v):
    """
    Solves Node Voltage vx (Top-Center Node).
    Nodes:
    - Left Source: V1
    - Center (vx): Top of R_share.
    - Right (vy): Top Right Corner.

    Topology Update based on Mesh Eq:
    Mesh 2 equation implies simple series loop.
    If DepSource is on Top Branch and Resistor R2 is Right Vertical.
    DepSource is 5Vx.
    Relationship between vx and vy (across DepSource):
    vy - vx = Gain? No.
    Mesh 2 KVL trace: Start at vx -> go Right through DepSource -> vy.
    If we encountered Negative Terminal first (Rise), then vy > vx.
    vy = vx + 5Vx.
    Since Vx = vx (referenced to ground), vy = vx + 5vx = 6vx.

    KCL at vy (Top Right):
    Current leaving left towards vx? (vy - vx)/R_source? No, DepSource has 0 resistance ideal.
    So vx and vy are "Supernode" connected by DepSource.
    KCL for Supernode (vx + vy):
    1. Leaving Left from vx: (vx - V1)/R1
    2. Leaving Down from vx: vx/R_sh
    3. Leaving Down from vy: vy/R2
    Sum = 0.

    Sub vy = (1 + k)vx:
    (vx - V1)/R1 + vx/R_sh + (1+k)vx/R2 = 0
    vx [1/R1 + 1/R_sh + (1+k)/R2] = V1/R1
    """
    R1, R2, R_sh = v['R1'], v['R2'], v['R_share']
    V1 = v['V1']
    k = v['k']

    # Conductance term
    G_total = (1/R1) + (1/R_sh) + ((1 + k)/R2)
    I_source = V1 / R1

    vx = I_source / G_total

    return {"v_nodes": {"vx": vx}}

def prompt_prob5(v):
    return f"""Analyze the Circuit with a Dependent Voltage Source.

**Components:**
- **Mesh 1 (Left):** Independent Voltage Source V1={v['V1']}V (Positive Up), Top Resistor R1={v['R1']}Ω. Shares Vertical Resistor R_share={v['R_share']}Ω with Mesh 2.
- **Mesh 2 (Right):** Shares R_share with Mesh 1. Top Branch contains **Dependent Voltage Source** (Negative Left, Positive Right). Right Branch contains Resistor R2={v['R2']}Ω.
- **Dependent Source:** Value = {v['k']} * Vx.
- **Control Variable:** Vx is the voltage across the shared resistor R_share (Positive at Top).

**Variables:**
- **i1, i2:** Mesh currents (Clockwise).
- **vx:** Voltage at the top-center node.

**Tasks:**
1. (KVL) Write mesh equations for i1, i2.
2. (KCL) Write the nodal equation for vx.
3. Solve for i1, i2, and vx."""

# --- CONSISTENCY CHECK ---
def check_prob5(kvl, kcl, v):
    i1, i2 = kvl['i_mesh']

    # Verify vx from Mesh
    # vx = (i1 - i2) * R_share
    vx_mesh = (i1 - i2) * v['R_share']
    vx_node = kcl['v_nodes']['vx']

    if abs(vx_mesh - vx_node) > 1e-3:
        print(f"FAIL: vx mismatch. Mesh: {vx_mesh:.4f} V, Nodal: {vx_node:.4f} V")
        return False

    print(f"PASS: Consistency Check. vx={vx_mesh:.4f}V")
    return True

# Uncomment the lines below to verify this block.
# # --- INITIALIZE ---
# prob5 = CircuitProblem("Prob5_Dependent", solve_prob5_kvl, solve_prob5_kcl, prompt_prob5)
# prob5._verify_consistency = check_prob5

# # --- GENERATE DATA ---
# # 1. Control (AAC PDF Values)
# # V1=3, R1=100, R_sh=200, R2=300 (Derived), k=5
# # Expected: i1=21.4mA, i2=17.1mA
# dataset.append(prob5.generate(
#     {'V1': 3, 'R1': 100, 'R_share': 200, 'R2': 300, 'k': 5},
#     "Control"
# ))

# # 2. Trap (High Gain)
# dataset.append(prob5.generate(
#     {'V1': 3, 'R1': 100, 'R_share': 200, 'R2': 300, 'k': 10},
#     "Trap"
# ))

# if __name__ == "__main__":
#     # Check AAC Match
#     i_ctrl = dataset[-2]['ground_truth']['mesh_currents']
#     print(f"AAC Check: i1={i_ctrl[0]*1000:.1f}mA (Exp 21.4), i2={i_ctrl[1]*1000:.1f}mA (Exp 17.1)")

In [ ]:
# depracated DO NOT RUN
# @title
# ==============================================================================
# 3. SAMPLE DATASET GENERATION (Using Verified Method) (depracated)
# ==============================================================================
if __name__ == "__main__":
    # 1. Initialize the Master List
    dataset = []

    # --- PROBLEM 1: SUPERMESH ---
    # Instantiate
    prob1 = CircuitProblem("Prob1_Supermesh", solve_prob1_kvl, solve_prob1_kcl, prompt_prob1)
    prob1._verify_consistency = check_prob1

    # Generate Control (Sadiku 3.62: 100V, 40V, 4mA, 4k, 8k, 2k)
    dataset.append(prob1.generate(
        {'V1': 100, 'V2': 40, 'Is': 0.004, 'R1': 4000, 'R2': 8000, 'R3': 2000},
        "Control"
    ))

    # Generate Control (Sadiku 3.62: 100V, 40V, 4A, 4, 8, 2)
    dataset.append(prob1.generate(
        {'V1': 100, 'V2': 40, 'Is': 4, 'R1': 4, 'R2': 8, 'R3': 2},
        "Control-2"
    ))
    # Generate Trap (Modified V1)
    dataset.append(prob1.generate(
        {'V1': 105, 'V2': 40, 'Is': 0.004, 'R1': 4000, 'R2': 8000, 'R3': 2000},
        "Trap"
    ))

    # --- PROBLEM 2: OPPOSING SOURCES ---
    prob2 = CircuitProblem("Prob2_Opposing", solve_prob2_kvl, solve_prob2_kcl, prompt_prob2)
    prob2._verify_consistency = check_prob2

    # Generate Control (Kuphaldt: 28V, 7V, 4Ω, 1Ω, 2Ω. Result: 5A, 1A)
    dataset.append(prob2.generate(
        {'V1': 28, 'V2': 7, 'R1': 4, 'R2': 1, 'R_share': 2},
        "Control"
    ))
    # Generate Trap (Reversed: V2=50V)
    dataset.append(prob2.generate(
        {'V1': 28, 'V2': 50, 'R1': 4, 'R2': 1, 'R_share': 2},
        "Trap"
    ))

    # --- PROBLEM 3: UNBALANCED BRIDGE ---
    prob3 = CircuitProblem("Prob3_Bridge", solve_prob3_kvl, solve_prob3_kcl, prompt_prob3)
    prob3._verify_consistency = check_prob3

    # Generate Control (Kuphaldt: 24V, 150/50 Top, 300/250 Bot, 100 Bridge)
    dataset.append(prob3.generate(
        {'V': 240, 'R_LT': 150, 'R_LB': 300, 'R_RT': 50, 'R_RB': 250, 'R_Br': 100},
        "Control"
    ))
    # Generate Trap (Wildly Unbalanced: R_RB=500)
    dataset.append(prob3.generate(
        {'V': 24, 'R_LT': 150, 'R_LB': 100, 'R_RT': 50, 'R_RB': 500, 'R_Br': 250},
        "Trap"
    ))

    # --- PROBLEM 4: RESISTIVE LADDER ---
    prob4 = CircuitProblem("Prob4_Ladder", solve_prob4_kvl, solve_prob4_kcl, prompt_prob4)
    prob4._verify_consistency = check_prob4

    # Generate Control (PDF Values: 52V, All 10k)
    dataset.append(prob4.generate(
        {'V': 52, 'R1': 10000, 'R2': 10000, 'R3': 10000, 'R4': 10000, 'R5': 10000, 'R6': 10000},
        "Control"
    ))
    # Generate Control (PDF Values: 52V, All 10)
    dataset.append(prob4.generate(
        {'V': 52, 'R1': 10, 'R2': 10, 'R3': 10, 'R4': 10, 'R5': 10, 'R6': 10},
        "Control2"
    ))
    # Generate Trap (Modified Source/Resistor)
    dataset.append(prob4.generate(
        {'V': 60, 'R1': 10000, 'R2': 10000, 'R3': 10000, 'R4': 12000, 'R5': 10000, 'R6': 10000},
        "Trap"
    ))

    # --- PROBLEM 5: DEPENDENT SOURCE ---
    prob5 = CircuitProblem("Prob5_Dependent", solve_prob5_kvl, solve_prob5_kcl, prompt_prob5)
    prob5._verify_consistency = check_prob5

    # Generate Control (AAC PDF: 3V, 100/200/100, k=5)
    dataset.append(prob5.generate(
        {'V1': 3, 'R1': 100, 'R_share': 200, 'R2': 300, 'k': 5},
        "Control"
    ))
    # Generate Trap (High Gain k=10)
    dataset.append(prob5.generate(
        {'V1': 3, 'R1': 100, 'R_share': 200, 'R2': 300, 'k': 10},
        "Trap"
    ))

    # --- EXPORT TO JSON ---
    filename = 'circuchain_full_dataset.json'
    with open(filename, 'w') as f:
        json.dump(dataset, f, indent=2)

    print(f"SUCCESS: Generated {len(dataset)} verified problems.")
    print(f"Saved to: {filename}")

Master dataset generation

In [ ]:
# @title
# ==============================================================================
# 3. MASTER DATASET GENERATION (Using Verified Method)
# ==============================================================================
if __name__ == "__main__":
    # 1. Initialize the Master List
    dataset = []


    # --- PROBLEM 1: SUPERMESH (10 VERIFIED VARIATIONS) ---
    prob1 = CircuitProblem("Prob1_Supermesh", solve_prob1_kvl, solve_prob1_kcl, prompt_prob1)
    prob1._verify_consistency = check_prob1

    # 1. CONTROL (Sadiku 3.62 Base)
    # Truth: i1=0.002 A, i2=0.006 A, i3=0.002 A, vx=92.0 V
    dataset.append(prob1.generate(
        {'V1': 100, 'V2': 40, 'Is': 0.004, 'R1': 4000, 'R2': 8000, 'R3': 2000},
        "Control-Base"
    ))

    # 2. CONTROL (Sadiku 3.62 Integers)
    # Truth: i1=2.0 A, i2=6.0 A, i3=2.0 A, vx=92.0 V
    dataset.append(prob1.generate(
        {'V1': 100, 'V2': 40, 'Is': 4, 'R1': 4, 'R2': 8, 'R3': 2},
        "Control-Integer"
    ))

    # 3. TRAP (Modified V1 - High Voltage)
    # Truth: i1=0.0025 A, i2=0.0065 A, i3=0.0015 A, vx=95.0 V
    dataset.append(prob1.generate(
        {'V1': 105, 'V2': 40, 'Is': 0.004, 'R1': 4000, 'R2': 8000, 'R3': 2000},
        "Trap-HighVolt-1"
    ))

    # 4. CONTROL (Integer Variation 1)
    # Truth: i1=1.0 A, i2=3.0 A, i3=1.0 A, vx=74.0 V
    dataset.append(prob1.generate(
        {'V1': 80, 'V2': 45, 'Is': 2, 'R1': 6, 'R2': 9, 'R3': 2},
        "Control-Var1"
    ))

    # 5. CONTROL (High Resistance)
    # Truth: i1=0.002692 A, i2=0.007692 A, i3=0.002692 A, vx=139.23 V
    dataset.append(prob1.generate(
        {'V1': 150, 'V2': 60, 'Is': 0.005, 'R1': 4000, 'R2': 9000, 'R3': 2000},
        "Control-HighRes"
    ))

    # 6. CONTROL (Asymmetric Resistors)
    # Truth: i1=0.6471 A, i2=2.1471 A, i3=0.1471 A, vx=56.76 V
    dataset.append(prob1.generate(
        {'V1': 60, 'V2': 25, 'Is': 1.5, 'R1': 5, 'R2': 14, 'R3': 2},
        "Control-Asym"
    ))

    # 7. CONTROL (Balanced Low Current)
    # Truth: i1=0.002850 A, i2=0.006350 A, i3=0.002850 A, vx=80.75 V
    dataset.append(prob1.generate(
        {'V1': 95, 'V2': 35, 'Is': 0.0035, 'R1': 5000, 'R2': 7000, 'R3': 2000},
        "Control-LowCurr"
    ))

    # 8. TRAP (Very High Voltage 1)
    # Truth: i1=11.2 A, i2=15.2 A, i3=7.2 A, vx=147.2 V
    dataset.append(prob1.generate(
        {'V1': 192, 'V2': 40, 'Is': 4, 'R1': 4, 'R2': 8, 'R3': 2},
        "Trap-HighVolt-2"
    ))

    # 9. TRAP (Very High Voltage 2)
    # Truth: i1=20.0 A, i2=24.8 A, i3=15.2 A, vx=218.0 V
    dataset.append(prob1.generate(
        {'V1': 298, 'V2': 50, 'Is': 4.8, 'R1': 4, 'R2': 8, 'R3': 2},
        "Trap-HighVolt-3"
    ))

    # 10. TRAP (Asymmetric High V)
    # Truth: i1=14.6154 A, i2=17.6154 A, i3=11.6154 A, vx=196.92 V
    dataset.append(prob1.generate(
        {'V1': 270, 'V2': 44, 'Is': 3, 'R1': 5, 'R2': 10, 'R3': 2},
        "Trap-Asym-HighV"
    ))

    # --- PROBLEM 2: OPPOSING SOURCES (10 VARIATIONS) ---
    prob2 = CircuitProblem("Prob2_Opposing", solve_prob2_kvl, solve_prob2_kcl, prompt_prob2)
    prob2._verify_consistency = check_prob2

    # User Control (Kuphaldt Base)
    # Truth: i1=5.0000 A, i2=1.0000 A, v1=8.0000 V
    dataset.append(prob2.generate({'V1': 28, 'V2': 7, 'R1': 4, 'R2': 1, 'R_share': 2}, "Control-Base"))

    # User Trap (Reversed Dominance)
    # Truth: i1=-1.1429 A, i2=-17.4286 A, v1=32.5714 V
    dataset.append(prob2.generate({'V1': 28, 'V2': 50, 'R1': 4, 'R2': 1, 'R_share': 2}, "Trap-Reversed"))

    # Generated Control-1
    # Truth: i1=6.4894 A, i2=1.8085 A, v1=14.0426 V
    dataset.append(prob2.generate({'V1': 40, 'V2': 5, 'R1': 4, 'R2': 5, 'R_share': 3}, "Control-Gen-1"))

    # Generated Control-2
    # Truth: i1=7.0000 A, i2=4.0000 A, v1=15.0000 V
    dataset.append(prob2.generate({'V1': 36, 'V2': 11, 'R1': 3, 'R2': 1, 'R_share': 5}, "Control-Gen-2"))

    # Generated Control-3
    # Truth: i1=6.4000 A, i2=-0.4000 A, v1=6.8000 V
    dataset.append(prob2.generate({'V1': 26, 'V2': 8, 'R1': 3, 'R2': 3, 'R_share': 1}, "Control-Gen-3"))

    # Generated Control-4
    # Truth: i1=7.1429 A, i2=2.4286 A, v1=9.4286 V
    dataset.append(prob2.generate({'V1': 38, 'V2': 7, 'R1': 4, 'R2': 1, 'R_share': 2}, "Control-Gen-4"))

    # Generated Trap-1
    # Truth: i1=5.4706 A, i2=-8.5882 A, v1=14.0588 V
    dataset.append(prob2.generate({'V1': 25, 'V2': 57, 'R1': 2, 'R2': 5, 'R_share': 1}, "Trap-Gen-1"))

    # Generated Trap-2
    # Truth: i1=-0.8750 A, i2=-15.9375 A, v1=30.1250 V
    dataset.append(prob2.generate({'V1': 24, 'V2': 62, 'R1': 7, 'R2': 2, 'R_share': 2}, "Trap-Gen-2"))

    # Generated Trap-3
    # Truth: i1=-0.5319 A, i2=-7.6809 A, v1=28.5957 V
    dataset.append(prob2.generate({'V1': 27, 'V2': 67, 'R1': 3, 'R2': 5, 'R_share': 4}, "Trap-Gen-3"))

    # Generated Trap-4
    # Truth: i1=0.1667 A, i2=-5.5417 A, v1=22.8333 V
    dataset.append(prob2.generate({'V1': 24, 'V2': 45, 'R1': 7, 'R2': 4, 'R_share': 4}, "Trap-Gen-4"))


    # --- PROBLEM 3: UNBALANCED BRIDGE (10 VARIATIONS) ---
    prob3 = CircuitProblem("Prob3_Bridge", solve_prob3_kvl, solve_prob3_kcl, prompt_prob3)
    prob3._verify_consistency = check_prob3

    # User Control (Kuphaldt Base)
    # Truth: i1=1.3609 A, i2=0.9379 A, i3=0.7724 A, va=176.5517 V, vb=193.1034 V
    dataset.append(prob3.generate({'V': 240, 'R_LT': 150, 'R_LB': 300, 'R_RT': 50, 'R_RB': 250, 'R_Br': 100}, "Control-Base"))

    # User Trap (Wildly Unbalanced)
    # Truth: i1=0.1571 A, i2=0.0749 A, i3=0.0405 A, va=11.6624 V, vb=20.2558 V
    dataset.append(prob3.generate({'V': 24, 'R_LT': 150, 'R_LB': 100, 'R_RT': 50, 'R_RB': 500, 'R_Br': 250}, "Trap-Unbalanced"))

    # Generated Control-1
    # Truth: i1=0.5971 A, i2=0.3944 A, i3=0.3301 A, va=73.4381 V, vb=81.8593 V
    dataset.append(prob3.generate({'V': 100, 'R_LT': 131, 'R_LB': 275, 'R_RT': 46, 'R_RB': 248, 'R_Br': 131}, "Control-Gen-1"))

    # Generated Control-2
    # Truth: i1=1.6050 A, i2=1.1051 A, i3=0.9617 A, va=211.0119 V, vb=230.8009 V
    dataset.append(prob3.generate({'V': 296, 'R_LT': 170, 'R_LB': 328, 'R_RT': 59, 'R_RB': 240, 'R_Br': 138}, "Control-Gen-2"))

    # Generated Control-3
    # Truth: i1=1.8851 A, i2=1.2966 A, i3=0.9433 A, va=189.2972 V, vb=217.9094 V
    dataset.append(prob3.generate({'V': 297, 'R_LT': 183, 'R_LB': 201, 'R_RT': 61, 'R_RB': 231, 'R_Br': 81}, "Control-Gen-3"))

    # Generated Control-4
    # Truth: i1=1.0122 A, i2=0.6353 A, i3=0.5605 A, va=152.2458 V, vb=161.9754 V
    dataset.append(prob3.generate({'V': 202, 'R_LT': 132, 'R_LB': 337, 'R_RT': 63, 'R_RB': 289, 'R_Br': 130}, "Control-Gen-4"))

    # Generated Trap-1
    # Truth: i1=0.3323 A, i2=0.1170 A, i3=0.0432 A, va=17.0611 V, vb=41.2214 V
    dataset.append(prob3.generate({'V': 50, 'R_LT': 153, 'R_LB': 59, 'R_RT': 75, 'R_RB': 955, 'R_Br': 327}, "Trap-Gen-1"))

    # Generated Trap-2
    # Truth: i1=0.1819 A, i2=0.0631 A, i3=0.0193 A, va=10.0790 V, vb=23.1605 V
    dataset.append(prob3.generate({'V': 26, 'R_LT': 134, 'R_LB': 62, 'R_RT': 45, 'R_RB': 1197, 'R_Br': 299}, "Trap-Gen-2"))

    # Generated Trap-3
    # Truth: i1=0.2419 A, i2=0.0950 A, i3=0.0397 A, va=16.5789 V, vb=38.0117 V
    dataset.append(prob3.generate({'V': 42, 'R_LT': 173, 'R_LB': 82, 'R_RT': 42, 'R_RB': 957, 'R_Br': 388}, "Trap-Gen-3"))

    # Generated Trap-4
    # Truth: i1=0.1385 A, i2=0.0539 A, i3=0.0166 A, va=10.6018 V, vb=24.7882 V
    dataset.append(prob3.generate({'V': 27, 'R_LT': 194, 'R_LB': 87, 'R_RT': 41, 'R_RB': 1492, 'R_Br': 380}, "Trap-Gen-4"))

    # --- PROBLEM 4: RESISTIVE LADDER (10 VARIATIONS) ---
    prob4 = CircuitProblem("Prob4_Ladder", solve_prob4_kvl, solve_prob4_kcl, prompt_prob4)
    prob4._verify_consistency = check_prob4

    # User Control (PDF Values: 52V, All 10k)
    # Truth: i1=0.003200 A, i2=0.001200 A, i3=0.000400 A, v1=20.0000 V, v2=8.0000 V, v3=4.0000 V
    dataset.append(prob4.generate({'V': 52, 'R1': 10000, 'R2': 10000, 'R3': 10000, 'R4': 10000, 'R5': 10000, 'R6': 10000}, "Control-10k"))

    # User Control 2 (PDF Values: 52V, All 10)
    # Truth: i1=3.200000 A, i2=1.200000 A, i3=0.400000 A, v1=20.0000 V, v2=8.0000 V, v3=4.0000 V
    dataset.append(prob4.generate({'V': 52, 'R1': 10, 'R2': 10, 'R3': 10, 'R4': 10, 'R5': 10, 'R6': 10}, "Control-10"))

    # User Trap (Modified Source/Resistor)
    # Truth: i1=0.003667 A, i2=0.001333 A, i3=0.000500 A, v1=23.3333 V, v2=10.0000 V, v3=5.0000 V
    dataset.append(prob4.generate({'V': 60, 'R1': 10000, 'R2': 10000, 'R3': 10000, 'R4': 12000, 'R5': 10000, 'R6': 10000}, "Trap-Modified"))

    # Generated Control-1
    # Truth: i1=0.004434 A, i2=0.002458 A, i3=0.000375 A, v1=7.1487 V, v2=2.6648 V, v3=1.0678 V
    dataset.append(prob4.generate({'V': 28, 'R1': 4703, 'R2': 3619, 'R3': 1824, 'R4': 1279, 'R5': 4261, 'R6': 2849}, "Control-Gen-1"))

    # Generated Control-2
    # Truth: i1=0.006822 A, i2=0.003035 A, i3=0.000714 A, v1=12.0474 V, v2=6.3236 V, v3=3.3875 V
    dataset.append(prob4.generate({'V': 43, 'R1': 4537, 'R2': 3181, 'R3': 1886, 'R4': 2725, 'R5': 4110, 'R6': 4742}, "Control-Gen-2"))

    # Generated Control-3
    # Truth: i1=0.016012 A, i2=0.005531 A, i3=0.001395 A, v1=23.3289 V, v2=7.1935 V, v3=2.3630 V
    dataset.append(prob4.generate({'V': 55, 'R1': 1978, 'R2': 2226, 'R3': 2917, 'R4': 1739, 'R5': 3463, 'R6': 1694}, "Control-Gen-3"))

    # Generated Trap-1
    # Truth: i1=0.088378 A, i2=0.029755 A, i3=0.028108 A, v1=58.6224 V, v2=28.8673 V, v3=0.7589 V
    dataset.append(prob4.generate({'V': 147, 'R1': 1000, 'R2': 1000, 'R3': 1000, 'R4': 17530, 'R5': 1000, 'R6': 27}, "Trap-Gen-1"))

    # Generated Trap-2
    # Truth: i1=0.108661 A, i2=0.036323 A, i3=0.034465 A, v1=72.3386 V, v2=36.0157 V, v3=1.5509 V
    dataset.append(prob4.generate({'V': 181, 'R1': 1000, 'R2': 1000, 'R3': 1000, 'R4': 19384, 'R5': 1000, 'R6': 45}, "Trap-Gen-2"))

    # Generated Trap-3
    # Truth: i1=0.075154 A, i2=0.025308 A, i3=0.024081 A, v1=49.8461 V, v2=24.5383 V, v3=0.4575 V
    dataset.append(prob4.generate({'V': 125, 'R1': 1000, 'R2': 1000, 'R3': 1000, 'R4': 19998, 'R5': 1000, 'R6': 19}, "Trap-Gen-3"))

    # Generated Trap-4
    # Truth: i1=0.085275 A, i2=0.028549 A, i3=0.026041 A, v1=56.7254 V, v2=28.1761 V, v3=2.1353 V
    dataset.append(prob4.generate({'V': 142, 'R1': 1000, 'R2': 1000, 'R3': 1000, 'R4': 11232, 'R5': 1000, 'R6': 82}, "Trap-Gen-4"))

    # --- PROBLEM 5: DEPENDENT SOURCE (10 VARIATIONS) ---
    prob5 = CircuitProblem("Prob5_Dependent", solve_prob5_kvl, solve_prob5_kcl, prompt_prob5)
    prob5._verify_consistency = check_prob5

    # User Control (AAC PDF Values)
    # Truth: i1=0.021429 A, i2=0.017143 A, vx=0.857143 V
    dataset.append(prob5.generate({'V1': 3, 'R1': 100, 'R_share': 200, 'R2': 300, 'k': 5}, "Control-AAC"))

    # User Trap (High Gain k=10)
    # Truth: i1=0.024194 A, i2=0.021290 A, vx=0.580645 V
    dataset.append(prob5.generate({'V1': 3, 'R1': 100, 'R_share': 200, 'R2': 300, 'k': 10}, "Trap-HighGain"))

    # Generated Control-1
    # Truth: i1=0.434584 A, i2=0.327576 A, vx=5.350407 V
    dataset.append(prob5.generate({'V1': 11, 'R1': 13, 'R_share': 50, 'R2': 49, 'k': 2}, "Control-Gen-1"))

    # Generated Control-2
    # Truth: i1=0.199630 A, i2=0.092421 A, vx=1.608133 V
    dataset.append(prob5.generate({'V1': 6, 'R1': 22, 'R_share': 15, 'R2': 87, 'k': 4}, "Control-Gen-2"))

    # Generated Control-3
    # Truth: i1=0.778190 A, i2=0.503970 A, vx=4.661726 V
    dataset.append(prob5.generate({'V1': 14, 'R1': 12, 'R_share': 17, 'R2': 37, 'k': 3}, "Control-Gen-3"))

    # Generated Control-4
    # Truth: i1=0.330343 A, i2=0.234910 A, vx=3.053834 V
    dataset.append(prob5.generate({'V1': 9, 'R1': 18, 'R_share': 32, 'R2': 65, 'k': 4}, "Control-Gen-4"))

    # Generated Trap-1
    # Truth: i1=1.353641 A, i2=1.311756 A, vx=2.219895 V
    dataset.append(prob5.generate({'V1': 32, 'R1': 22, 'R_share': 53, 'R2': 22, 'k': 12}, "Trap-Gen-1"))

    # Generated Trap-2
    # Truth: i1=0.521221 A, i2=0.503172 A, vx=1.660468 V
    dataset.append(prob5.generate({'V1': 10, 'R1': 16, 'R_share': 92, 'R2': 33, 'k': 9}, "Trap-Gen-2"))

    # Generated Trap-3
    # Truth: i1=1.530426 A, i2=1.452638 A, vx=5.678496 V
    dataset.append(prob5.generate({'V1': 47, 'R1': 27, 'R_share': 73, 'R2': 43, 'k': 10}, "Trap-Gen-3"))

    # Generated Trap-4
    # Truth: i1=0.643701 A, i2=0.627618 A, vx=1.608271 V
    dataset.append(prob5.generate({'V1': 28, 'R1': 41, 'R_share': 100, 'R2': 41, 'k': 15}, "Trap-Gen-4"))


# --- EXPORT TO JSON ---
    filename = 'circuchain_full_dataset.json'
    with open(filename, 'w') as f:
        json.dump(dataset, f, indent=2)

    print(f"SUCCESS: Generated {len(dataset)} verified problems.")
    print(f"Saved to: {filename}")

Now we will verify the problems using the spice script.

In [ ]:
# @title
import json
import os

# ==============================================================================
# SPICE GENERATORS FOR EACH TOPOLOGY
# ==============================================================================

def gen_spice_prob1(values, truth):
    # Prob 1: Supermesh
    # Topology: V1-R1-Is-R2-DepSrc-R3-V2
    return f"""* Prob 1: Supermesh (Auto-Generated)
* EXPECTED TRUTH:
* i1 (I(R1)) = {truth['mesh_currents'][0]:.6f} A
* i2 (I(R2)) = {truth['mesh_currents'][1]:.6f} A
* i3 (I(R3)) = {truth['mesh_currents'][2]:.6f} A
* vx (V(2))  = {truth['node_voltages'].get('vx', 0):.4f} V
* vy (V(3))  = {truth['node_voltages'].get('vy', 0):.4f} V

* --- MESH 1 ---
V1 1 0 DC {values['V1']}
R1 1 2 {values['R1']}

* --- SHARED 1-2 (Current Source UP) ---
I_source 0 2 DC {values['Is']}

* --- MESH 2 ---
R2 2 3 {values['R2']}

* --- SHARED 2-3 (Dep Current Source DOWN) ---
* Value = 2 * i1. i1 is current through R1.
* Using Behavioral Source for LTSpice robustness.
B_dep 3 0 I=2*I(R1)

* --- MESH 3 ---
R3 3 4 {values['R3']}
V2 4 0 DC {values['V2']}

.save all
.op
.print op v(2) v(3) I(R1) I(R2) I(R3)
.end
"""

def gen_spice_prob2(values, truth):
    # Prob 2: Opposing Sources
    return f"""* Prob 2: Opposing Sources (Auto-Generated)
* EXPECTED TRUTH:
* i1 (I(R1)) = {truth['mesh_currents'][0]:.6f} A
* i2 (I(R2)) = {truth['mesh_currents'][1]:.6f} A
* v1 (V(2))  = {truth['node_voltages'].get('v1', 0):.4f} V

* --- LEFT LOOP ---
V1 1 0 DC {values['V1']}
R1 1 2 {values['R1']}

* --- SHARED ---
R_share 2 0 {values['R_share']}

* --- RIGHT LOOP ---
R2 2 3 {values['R2']}
V2 3 0 DC {values['V2']}

.save all
.op
.print op v(2) I(R1) I(R2)
.end
"""

def gen_spice_prob3(values, truth):
    # Prob 3: Unbalanced Bridge
    return f"""* Prob 3: Unbalanced Bridge (Auto-Generated)
* EXPECTED TRUTH:
* i1 (I(V_src)) = {truth['mesh_currents'][0]:.6f} A (Approx, Mesh 1 is loop)
* va (V(2))     = {truth['node_voltages'].get('va', 0):.4f} V
* vb (V(3))     = {truth['node_voltages'].get('vb', 0):.4f} V

* --- Source ---
V_src 1 0 DC {values['V']}

* --- Left Leg ---
R_LT 1 2 {values['R_LT']}
R_LB 2 0 {values['R_LB']}

* --- Right Leg ---
R_RT 1 3 {values['R_RT']}
R_RB 3 0 {values['R_RB']}

* --- Bridge ---
R_Br 2 3 {values['R_Br']}

.save all
.op
.print op v(2) v(3) I(V_src)
.end
"""

def gen_spice_prob4(values, truth):
    # Prob 4: Resistive Ladder
    return f"""* Prob 4: Resistive Ladder (Auto-Generated)
* EXPECTED TRUTH:
* i1 (I(R1)) = {truth['mesh_currents'][0]:.6f} A
* i2 (I(R3)) = {truth['mesh_currents'][1]:.6f} A
* i3 (I(R5)) = {truth['mesh_currents'][2]:.6f} A
* v1 (V(2))  = {truth['node_voltages'].get('v1', 0):.4f} V
* v2 (V(3))  = {truth['node_voltages'].get('v2', 0):.4f} V
* v3 (V(4))  = {truth['node_voltages'].get('v3', 0):.4f} V

* --- Mesh 1 ---
V1 1 0 DC {values['V']}
R1 1 2 {values['R1']}
R2 2 0 {values['R2']}

* --- Mesh 2 ---
R3 2 3 {values['R3']}
R4 3 0 {values['R4']}

* --- Mesh 3 ---
R5 3 4 {values['R5']}
R6 4 0 {values['R6']}

.save all
.op
.print op v(2) v(3) v(4) I(R1) I(R3) I(R5)
.end
"""

def gen_spice_prob5(values, truth):
    # Prob 5: Dependent Source
    return f"""* Prob 5: Dependent Source (Auto-Generated)
* EXPECTED TRUTH:
* i1 (I(R1)) = {truth['mesh_currents'][0]:.6f} A
* i2 (I(R2)) = {truth['mesh_currents'][1]:.6f} A
* vx (V(2))  = {truth['node_voltages'].get('vx', 0):.4f} V

* --- Mesh 1 ---
V1 1 0 DC {values['V1']}
R1 1 2 {values['R1']}
R_share 2 0 {values['R_share']}

* --- Mesh 2 ---
* VCVS (Voltage Controlled Voltage Source)
* E_dep N+ N- Control+ Control- Gain
* N+ is Node 3, N- is Node 2 (Top Branch)
* Control is Voltage across R_share (Node 2 to 0)
E_dep 3 2 2 0 {values['k']}

R2 3 0 {values['R2']}

.save all
.op
.print op v(2) I(R1) I(R2)
.end
"""

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

def generate_all_spice_files(dataset_list, output_dir="spice_verification"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Generating SPICE files in '{output_dir}/'...")

    count = 0
    for problem in dataset_list:
        pid = problem['id']
        vals = problem['values']
        truth = problem['ground_truth']

        content = ""
        if "Prob1" in pid:
            content = gen_spice_prob1(vals, truth)
        elif "Prob2" in pid:
            content = gen_spice_prob2(vals, truth)
        elif "Prob3" in pid:
            content = gen_spice_prob3(vals, truth)
        elif "Prob4" in pid:
            content = gen_spice_prob4(vals, truth)
        elif "Prob5" in pid:
            content = gen_spice_prob5(vals, truth)

        if content:
            filename = f"{output_dir}/{pid}.cir"
            with open(filename, 'w') as f:
                f.write(content)
            count += 1

    print(f"Success! Generated {count} .cir files.")

    # Zip them for easy download in Colab
    os.system(f"zip -r {output_dir}.zip {output_dir}")
    print(f"Zipped archive created: {output_dir}.zip")

# --- Run ---
if __name__ == "__main__":
    # Assumes 'dataset' variable exists from previous step
    # If loading from file:
    # with open('circuchain_dataset_100.json', 'r') as f: dataset = json.load(f)
    generate_all_spice_files(dataset)

Generating SPICE files in 'spice_verification/'...
Success! Generated 50 .cir files.
Zipped archive created: spice_verification.zip


Now we will start evaluating and saving student model responses.

In [1]:
import os
import getpass
from google.colab import files

# 1. Enter OpenAI API Key securely
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")
# 1. Enter Anthropic API Key securely
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Antropic API Key: ")

# 2. Upload Dataset (If not already present)
# if not os.path.exists('circuchain_full_dataset.json'):
#     print("Upload 'circuchain_full_dataset.json' now:")
#     uploaded = files.upload()
# else:
#     print("Dataset 'circuchain_full_dataset.json' found.")

Enter your Antropic API Key: ··········


In [ ]:
# @title
import asyncio
import json
import time
import nest_asyncio
from openai import AsyncOpenAI

# Patch asyncio to allow nested loops in Colab
nest_asyncio.apply()

# ==============================================================================
# CONFIGURATION
# ==============================================================================
INPUT_FILE = 'circuchain_full_dataset.json'
OUTPUT_FILE = 'circuchain_llm_responses_o1.json'
MAX_CONCURRENT_REQUESTS = 5  # Adjust based on your Tier (5 for Tier 1, 50+ for Tier 4)
MODEL_NAME = ""         # Or "gpt-4-turbo", "o1-mini", etc.

# ==============================================================================
# SOLVER CLASS
# ==============================================================================
class CircuitSolver:
    def __init__(self):
        self.client = AsyncOpenAI() # Uses env var set in Cell 2
        self.results = []

    async def solve_single_problem(self, problem, semaphore):
        """
        Sends one problem to the LLM.
        """
        async with semaphore:
            print(f"Solving {problem['id']}...")
            start_time = time.time()

            try:
                response = await self.client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "system",
                            "content": "You are an expert electrical engineering AI. Solve the circuit problem step-by-step. Clearly define your Mesh/Nodal equations before solving."
                        },
                        {"role": "user", "content": problem['prompt']}
                    ],
                    # temperature=0.0 # Greedy decoding for consistency
                )
                answer_text = response.choices[0].message.content
            except Exception as e:
                answer_text = f"API_ERROR: {str(e)}"

            duration = time.time() - start_time

            # Package the Result
            return {
                "id": problem['id'],
                "model": MODEL_NAME,
                "prompt": problem['prompt'],
                "ground_truth": problem['ground_truth'], # Carry over truth for grading
                "response": answer_text,
                "latency": round(duration, 2)
            }

    async def run(self):
        # 1. Load Data
        try:
            with open(INPUT_FILE, 'r') as f:
                dataset = json.load(f)
            print(f"Loaded {len(dataset)} problems.")
        except FileNotFoundError:
            print("Error: Dataset file not found. Please run the generation script first.")
            return

        # 2. Parallel Execution
        semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
        tasks = [self.solve_single_problem(p, semaphore) for p in dataset]

        print(f"Starting parallel execution on {MODEL_NAME}...")
        self.results = await asyncio.gather(*tasks)

        # 3. Save Results
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(self.results, f, indent=2)

        print(f"\nCOMPLETED. {len(self.results)} responses saved to '{OUTPUT_FILE}'.")

        # 4. Preview one response
        if self.results:
            print(f"\n--- Sample Response ({self.results[0]['id']}) ---")
            print(self.results[0]['response'][:500] + "...")

# ==============================================================================
# EXECUTION
# ==============================================================================
solver = CircuitSolver()
await solver.run()

In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 14.2 MB/s eta 0:00:00


In [ ]:
# @title
import asyncio
import json
import time
import os
import nest_asyncio

# API Clients
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic
import google.generativeai as genai

# Patch for Colab
nest_asyncio.apply()

# ==============================================================================
# 1. GLOBAL CONVENTIONS & PROMPT ENGINEERING
# ==============================================================================
def construct_final_prompt(raw_problem, method_type):
    """
    Constructs the exact input string sent to the model.
    Injects strict conventions to normalize model behavior.
    """

    # 1. The Physics Constitution (Global Assumptions)
    conventions = """
--- GLOBAL CIRCUIT CONVENTIONS ---
1. MESH CURRENTS: Always assume CLOCKWISE direction for all loops unless explicitly drawn otherwise.
2. NODAL VOLTAGES: Always measure with respect to a common GROUND (0V) at the bottom of the circuit.
3. PASSIVE SIGN CONVENTION: Current enters the Positive (+) terminal of a passive component.
4. DEPENDENT SOURCES: If a dependency parameter (like 'ix' or 'vx') is not explicitly defined in the text, infer it from the standard node locations described.
"""

    # 2. Method Constraints
    if method_type == "KVL":
        constraint = """
--- TASK CONSTRAINT: MESH ANALYSIS (KVL) ---
You MUST solve this using Mesh Analysis (Kirchhoff's Voltage Law).
1. Set up KVL equations for each loop.
2. Solve for mesh currents first.
3. Calculate node voltages derived from these currents.
"""
    elif method_type == "KCL":
        constraint = """
--- TASK CONSTRAINT: NODAL ANALYSIS (KCL) ---
You MUST solve this using Nodal Analysis (Kirchhoff's Current Law).
1. Identify essential nodes and reference ground.
2. Set up KCL equations for each node.
3. Solve for node voltages first.
4. Calculate branch currents derived from these voltages.
"""

    # 3. Final Assembly
    return f"""You are an expert Electrical Engineering Research Assistant.
Your goal is to solve the circuit problem below with high precision.

{conventions}

--- PROBLEM DESCRIPTION ---
{raw_problem}

{constraint}

--- REQUIRED OUTPUT FORMAT ---
You must end your response with a clear list of the final values.
Use strictly this format: "variable = value"
Example:
i1 = 0.005
vx = 12.0
"""

# ==============================================================================
# 2. MODEL HANDLERS (The "Drivers")
# ==============================================================================

class BaseModelHandler:
    def __init__(self, model_name, api_key):
        self.model_name = model_name
        self.api_key = api_key

    async def generate(self, prompt, system_msg="You are a helpful engineer."):
        raise NotImplementedError

# --- OPENAI HANDLER (GPT-4o, o1) ---
class OpenAIHandler(BaseModelHandler):
    def __init__(self, model_name, api_key):
        super().__init__(model_name, api_key)
        self.client = AsyncOpenAI(api_key=self.api_key)

    async def generate(self, prompt, system_msg=""):
        try:
            # o1 models do not support system messages or temperature
            if "o1" in self.model_name:
                response = await self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": prompt}]
                )
            else:
                response = await self.client.chat.completions.create(
                    model=self.model_name,
                    temperature=0.0,
                    messages=[
                        {"role": "system", "content": system_msg},
                        {"role": "user", "content": prompt}
                    ]
                )
            return response.choices[0].message.content
        except Exception as e:
            return f"ERROR: {str(e)}"

# --- ANTHROPIC HANDLER (Claude) ---
class AnthropicHandler(BaseModelHandler):
    def __init__(self, model_name, api_key):
        super().__init__(model_name, api_key)
        self.client = AsyncAnthropic(api_key=self.api_key)

    async def generate(self, prompt, system_msg=""):
        try:
            response = await self.client.messages.create(
                model=self.model_name,
                max_tokens=2000,
                temperature=0.0,
                system=system_msg,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        except Exception as e:
            return f"ERROR: {str(e)}"

# --- GOOGLE HANDLER (Gemini) ---
class GeminiHandler(BaseModelHandler):
    def __init__(self, model_name, api_key):
        super().__init__(model_name, api_key)
        genai.configure(api_key=self.api_key)
        self.model = genai.GenerativeModel(self.model_name)

    async def generate(self, prompt, system_msg=""):
        try:
            # Gemini combines system/user slightly differently, simpler to prepend
            full_prompt = f"{system_msg}\n\n{prompt}"
            # Run in executor because Gemini SDK is synchronous
            response = await asyncio.to_thread(
                self.model.generate_content,
                full_prompt,
                generation_config=genai.types.GenerationConfig(temperature=0.0)
            )
            return response.text
        except Exception as e:
            return f"ERROR: {str(e)}"

# --- OPEN-ROUTER / LLAMA HANDLER ---
# Use this for Llama 3 via DeepInfra/Together/OpenRouter
class OpenRouterHandler(OpenAIHandler):
    def __init__(self, model_name, api_key, base_url):
        super().__init__(model_name, api_key)
        self.client = AsyncOpenAI(api_key=self.api_key, base_url=base_url)

# ==============================================================================
# 3. BENCHMARK ENGINE
# ==============================================================================
INPUT_FILE = 'circuchain_full_dataset.json'

async def run_benchmark(provider_key, handler_class, model_name, extra_args={}):
    """
    Runs the full dataset on a specific model provider.
    """
    api_key = os.getenv(provider_key)
    if not api_key and "base_url" not in extra_args: # Llama might use custom url
        print(f"⚠️ SKIPPING {model_name}: API Key {provider_key} not found.")
        return

    print(f"\n🚀 STARTING RUN: {model_name}")

    # Initialize Handler
    handler = handler_class(model_name, api_key, **extra_args) if extra_args else handler_class(model_name, api_key)

    # Load Data
    with open(INPUT_FILE, 'r') as f: dataset = json.load(f)

    output_filename = f"responses_{model_name.replace('/', '_')}.json"
    results = []

    # Semaphore for Rate Limiting (Adjust per provider)
    limit = 2 if "o1" in model_name else 2
    sem = asyncio.Semaphore(limit)

    async def process_task(problem, method):
        async with sem:
            prompt = construct_final_prompt(problem['prompt'], method)
            t0 = time.time()
            resp = await handler.generate(prompt, "You are a rigorous engineering AI.")
            dt = time.time() - t0

            print(f"  Finished {problem['id']} [{method}] in {dt:.2f}s")
            return {
                "id": problem['id'],
                "method": method,
                "model": model_name,
                "prompt_full": prompt,
                "response": resp,
                "latency": dt,
                "ground_truth": problem['ground_truth']
            }

    # Create Tasks (Fork KVL/KCL)
    tasks = []
    for p in dataset:
        tasks.append(process_task(p, "KVL"))
        tasks.append(process_task(p, "KCL"))

    results = await asyncio.gather(*tasks)

    # Save
    with open(output_filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✅ Saved {len(results)} results to {output_filename}")

# ==============================================================================
# 4. EXECUTION CONFIG (IEEE SoutheastCon Suite)
# ==============================================================================
async def main():
    # 1. The Baseline (Expect Low Performance)
    # Serves as the "Control Group" - proves the problems are hard.
    # await run_benchmark("OPENAI_API_KEY", OpenAIHandler, "gpt-4o-mini")

    # 2. The Standard (Expect Mixed Performance)
    # The current industry standard. Shows typical user experience.
    await run_benchmark("OPENAI_API_KEY", OpenAIHandler, "gpt-4o")

    # 3. The Specialist (Expect High Performance)
    # Represents "Reasoning Models" - the solution to the problem.
    # We use o1-mini because it is cost-effective and optimized for STEM.
    # await run_benchmark("OPENAI_API_KEY", OpenAIHandler, "o1-mini")

    # 4. The Challenger (Expect High Constraint Adherence)
    # Claude is excellent at following negative constraints ("Do NOT use...").
    # Great for checking if it handles the "Clockwise" rule better than o1.
    # await run_benchmark("ANTHROPIC_API_KEY", AnthropicHandler, "claude-opus-4-5-20251101")

if __name__ == "__main__":
    # Verify keys are loaded
    if not os.getenv("OPENAI_API_KEY"):
        print("❌ Error: OPENAI_API_KEY is missing.")
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("⚠️ Warning: ANTHROPIC_API_KEY is missing (Claude will be skipped).")

    asyncio.run(main())


🚀 STARTING RUN: gpt-4o
  Finished Prob5_Dependent_Control-AAC [KVL] in 11.69s
  Finished Prob5_Dependent_Control-AAC [KCL] in 17.46s
  Finished Prob5_Dependent_Trap-HighGain [KVL] in 13.77s
  Finished Prob5_Dependent_Trap-HighGain [KCL] in 13.92s
  Finished Prob5_Dependent_Control-Gen-1 [KVL] in 11.41s
  Finished Prob5_Dependent_Control-Gen-1 [KCL] in 14.29s
  Finished Prob5_Dependent_Control-Gen-2 [KVL] in 12.39s
  Finished Prob5_Dependent_Control-Gen-2 [KCL] in 12.95s
  Finished Prob5_Dependent_Control-Gen-3 [KVL] in 11.34s
  Finished Prob5_Dependent_Control-Gen-4 [KVL] in 10.74s
  Finished Prob5_Dependent_Control-Gen-3 [KCL] in 13.97s
  Finished Prob5_Dependent_Control-Gen-4 [KCL] in 14.82s
  Finished Prob5_Dependent_Trap-Gen-1 [KVL] in 13.91s
  Finished Prob5_Dependent_Trap-Gen-1 [KCL] in 13.66s
  Finished Prob5_Dependent_Trap-Gen-2 [KVL] in 14.14s
  Finished Prob5_Dependent_Trap-Gen-2 [KCL] in 11.71s
  Finished Prob5_Dependent_Trap-Gen-3 [KVL] in 13.79s
  Finished Prob5_Dependent

In [ ]:
try:
    files.download('circuchain_llm_responses_o1.json')
except Exception as e:
    print("Download failed (running locally?):", e)
    print("File is saved as 'circuchain_llm_responses.json' in the runtime.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now we will evaluate the resposes from the LLM solver, using LLM as a judge.

In [ ]:
# @title
import json
import re

INPUT_FILE = 'circuchain_llm_responses.json'

class CircuitJudge:
    def __init__(self):
        self.results = []

    def normalize_value(self, val_str):
        if not val_str: return None
        val_str = val_str.lower().strip()

        multiplier = 1.0
        if 'm' in val_str: multiplier = 0.001
        elif 'k' in val_str: multiplier = 1000.0

        # Extract number using regex that handles negatives and decimals
        nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", val_str)
        if not nums: return None

        # Return the last number found in the string (often the value)
        return float(nums[-1]) * multiplier

    def extract_answers(self, text, variables):
        found = {}
        # Normalize text to lower case for easier matching
        text_lower = text.lower()

        for var in variables:
            var_lower = var.lower()

            # Robust Patterns:
            # 1. "i1 = 5 mA"
            # 2. "i1 is 5 mA"
            # 3. "i1: 5 mA"
            # 4. "value of i1 is 5 mA"
            # 5. Boxed LaTeX: "\boxed{5}"

            # Try specific variable match first
            patterns = [
                rf"{var_lower}\s*[:=]\s*([-+]?[\d\.]+)\s*([mk]?[aAvV]?)",  # i1 = 5mA
                rf"{var_lower}\s*is\s*([-+]?[\d\.]+)\s*([mk]?[aAvV]?)",    # i1 is 5mA
                rf"boxed\{{\s*([-+]?[\d\.]+)\s*([mk]?[aAvV]?)\s*\}}"        # \boxed{5}
            ]

            val = None
            for pat in patterns:
                match = re.search(pat, text_lower)
                if match:
                    num_str = match.group(1)
                    unit_str = match.group(2)
                    val = self.normalize_value(num_str + unit_str)
                    break # Stop if found

            found[var] = val

        return found

    def evaluate(self):
        with open(INPUT_FILE, 'r') as f:
            responses = json.load(f)

        print(f"{'PROBLEM ID':<30} | {'VAR':<5} | {'TRUTH':<10} | {'PREDICT':<10} | {'STATUS'}")
        print("-" * 80)

        for item in responses:
            pid = item['id']
            truth_map = item['ground_truth']['verifier_map']

            # Extract
            student_vals = self.extract_answers(item['response'], truth_map.keys())

            for var, true_val in truth_map.items():
                pred_val = student_vals.get(var)

                status = "MISSING"
                if pred_val is not None:
                    # Tolerance check (1% or 0.001 absolute)
                    if abs(pred_val - true_val) < 1e-4 or abs((pred_val - true_val)/(true_val+1e-9)) < 0.01:
                        status = "PASS"
                    else:
                        status = "FAIL"

                print(f"{pid:<30} | {var:<5} | {true_val:<10.4f} | {str(pred_val):<10} | {status}")

            # Debug: Print snippets of missing ones
            if any(v is None for v in student_vals.values()):
                print(f"   [DEBUG] Raw Response Snippet: {item['response'][:100]}...")
            print("-" * 80)

if __name__ == "__main__":
    judge = CircuitJudge()
    judge.evaluate()

Judge Step 1: GPT 4o based value extractor

In [10]:
import asyncio
import json
import os
import nest_asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Finds response files
INPUT_FILES = [f for f in os.listdir('.') if f.startswith('responses_') and f.endswith('.json')]
EXTRACTOR_MODEL = "gpt-4o"
API_KEY = os.getenv("OPENAI_API_KEY")
client = AsyncOpenAI(api_key=API_KEY)

# ==============================================================================
# ROBUST EXTRACTOR LOGIC
# ==============================================================================
async def extract_data_robust(entry, debug=False):
    required_vars = list(entry['ground_truth']['verifier_map'].keys())

    # Improved Prompt: Explicitly handles variable naming variations
    system_prompt = """You are a robust Data Extraction Engine for Engineering.
    Your Goal: Extract final numerical values for circuit variables.

    CRITICAL RULES:
    1. FUZZY MATCHING: If I ask for "i1", accept "I1", "i_1", "I_mesh_1", "Current 1", etc.
    2. UNITS: Convert all to base units (mA -> 0.001, kOhm -> 1000).
    3. FORMAT: Return JSON: {"variable": float_value}.
    4. If a value is definitely not found, return null.
    """

    user_prompt = f"""
    TARGET VARIABLES: {', '.join(required_vars)}

    --- STUDENT RESPONSE TEXT ---
    {entry['response']}

    --- END TEXT ---

    Extract the values into JSON.
    """

    try:
        response = await client.chat.completions.create(
            model=EXTRACTOR_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        data = json.loads(content)

        # Cleanup: Ensure all keys exist
        cleaned = {k: data.get(k) for k in required_vars}

        if debug:
            print(f"\n--- DEBUG SAMPLE ({entry['id']}) ---")
            print(f"Targets: {required_vars}")
            print(f"Raw LLM Output: {content}")
            print(f"Cleaned Output: {cleaned}")
            print("-----------------------------------\n")

        return cleaned

    except Exception as e:
        print(f"Extraction Error {entry['id']}: {e}")
        return {k: None for k in required_vars}

async def process_file(filename):
    print(f"📂 Processing {filename}...")
    try:
        with open(filename, 'r') as f: data = json.load(f)
    except:
        print("Error reading file.")
        return

    # Check if data is empty
    if not data:
        print(f"⚠️  File {filename} is empty!")
        return

    # Check if responses are empty (Did the benchmark fail?)
    empty_responses = sum(1 for x in data if not x.get('response'))
    if empty_responses == len(data):
        print(f"❌ CRITICAL: All responses in {filename} are empty strings. Rerun Benchmark.")
        return

    # Process
    sem = asyncio.Semaphore(2)
    results = []

    # Run the first one in DEBUG mode so you can see what's happening
    first_task = await extract_data_robust(data[0], debug=True)
    results.append({**data[0], "extracted_values": first_task})

    async def worker(row):
        async with sem:
            vals = await extract_data_robust(row)
            await asyncio.sleep(3)
            return {**row, "extracted_values": vals}

    # Run the rest
    tasks = [worker(row) for row in data[1:]]
    results += await asyncio.gather(*tasks)

    outfile = filename.replace("responses_", "extracted_")
    with open(outfile, 'w') as f: json.dump(results, f, indent=2)
    print(f"✅ Saved to {outfile}")

async def main():
    if not INPUT_FILES:
        print("No input files found.")
        return
    await asyncio.gather(*[process_file(f) for f in INPUT_FILES])

if __name__ == "__main__":
    if not API_KEY:
        print("Please set OPENAI_API_KEY")
    else:
        asyncio.run(main())

📂 Processing responses_gpt-4o-mini.json...

--- DEBUG SAMPLE (Prob1_Supermesh_Control-Base) ---
Targets: ['i1', 'i2', 'i3', 'vx', 'vy']
Raw LLM Output: {
    "i1": 0.0071,
    "i2": 0.00215,
    "i3": 0.0123,
    "vx": 71.6,
    "vy": 17.2
}
Cleaned Output: {'i1': 0.0071, 'i2': 0.00215, 'i3': 0.0123, 'vx': 71.6, 'vy': 17.2}
-----------------------------------

✅ Saved to extracted_gpt-4o-mini.json


Judge Step 2: Python verifier for final answer

In [ ]:
import json
import os
import pandas as pd
import glob

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Add all your actual filenames here
INPUT_FILES = [
    "extracted_gpt-4o-mini.json",
    "extracted_gpt-4o.json",
    "extracted_o4-mini.json",
    "extracted_claude-opus-4-5.json",
    "extracted_gpt-5.json"
]
TOLERANCE = 0.05  # 5% margin

# ==============================================================================
# GRADING LOGIC
# ==============================================================================
def grade_problem(truth, extracted):
    if not extracted or all(v is None for v in extracted.values()):
        return "ERR_MISSING", {}

    errors = {}
    pass_cnt = 0
    sign_err_cnt = 0
    total = len(truth)

    for var, true_val in truth.items():
        pred_val = extracted.get(var)

        if pred_val is None:
            errors[var] = "MISSING"
            continue

        # Check Magnitude (Physics)
        diff = abs(pred_val - true_val)
        abs_true = abs(true_val)

        is_mag_correct = False
        if abs_true < 1e-6:
            if diff < 1e-5: is_mag_correct = True
        else:
            if (diff / abs_true) <= TOLERANCE: is_mag_correct = True

        if is_mag_correct:
            pass_cnt += 1
            errors[var] = "PASS"
        else:
            # Check Sign (Convention)
            mag_diff = abs(abs(pred_val) - abs_true)
            if abs_true > 1e-6 and (mag_diff / abs_true) <= TOLERANCE:
                sign_err_cnt += 1
                errors[var] = "ERR_SIGN"
            else:
                errors[var] = f"ERR_VAL (Got {pred_val}, Exp {true_val})"

    # Final Classification
    if pass_cnt == total:
        return "PASS", errors
    elif sign_err_cnt > 0 and (pass_cnt + sign_err_cnt == total):
        return "ERR_SIGN", errors # Correct Physics, Wrong Convention
    else:
        return "ERR_VAL", errors # Wrong Physics

# ==============================================================================
# RUNNER
# ==============================================================================
all_results = []

for fname in INPUT_FILES:
    if not os.path.exists(fname):
        print(f"⚠️ Missing: {fname}")
        continue

    # Clean Model Name
    model_name = fname.replace("extracted_", "").replace(".json", "")
    if "claude" in model_name: model_name = "Claude Opus 4.5"
    if "gpt-4o-mini" in model_name: model_name = "GPT-4o Mini"
    if "o4-mini" in model_name: model_name = "o4-mini"

    with open(fname, 'r') as f:
        data = json.load(f)

    print(f"Processing {model_name}...")

    for entry in data:
        truth = entry['ground_truth']['verifier_map']
        extracted = entry.get('extracted_values', {})
        grade, details = grade_problem(truth, extracted)

        all_results.append({
            "Model": model_name,
            "ID": entry['id'],
            "Type": "TRAP" if "Trap" in entry['id'] else "CONTROL",
            "Method": entry.get('method', 'Unknown'),
            "Grade": grade,
            "Details": details,
            "Prompt": entry.get('prompt_full', ''),
            "Response": entry.get('response', ''),
            "GroundTruth": truth # Needed for Step 3
        })

# Save Master File for Step 3
with open("graded_master_dataset.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\n✅ Saved 'graded_master_dataset.json' (Use this for Step 3)")

# Generate Summary Table
df = pd.DataFrame(all_results)
summary = df.pivot_table(
    index="Model", columns="Type", values="Grade",
    aggfunc=lambda x: (x == "PASS").mean() * 100
)
summary["Dropoff"] = summary["CONTROL"] - summary["TRAP"]
print("\n=== ACCURACY TABLE ===")
print(summary.round(1))
summary.to_csv("final_accuracy_table.csv")

Judge Step 3: GPT 5 based reasoning judge.

In [12]:
import asyncio
import json
import os
import nest_asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()

# ==============================================================================
# CONFIGURATION
# ==============================================================================
INPUT_FILE = "graded_master_dataset.json"
OUTPUT_FILE = "final_diagnosis_gpt5.json"
JUDGE_MODEL = "gpt-5" # Use the SOTA model
API_KEY = os.getenv("OPENAI_API_KEY")

client = AsyncOpenAI(api_key=API_KEY)

# ==============================================================================
# JUDGE PROMPT
# ==============================================================================
async def judge_entry(entry, sem):
    async with sem:
        if not entry.get('Response'):
            return {**entry, "Diagnosis": "NO_RESPONSE"}

        math_grade = entry['Grade']

        system_prompt = """You are an Expert Engineering Professor diagnosing student errors.

        INPUT DATA:
        1. Problem Statement & Required Method.
        2. Student's Full Response.
        3. Ground Truth Values.
        4. AUTOMATED MATH GRADE: This tells you if the final numbers were correct.
        (Note: The automated grade allows for small floating-point/rounding differences. Trust it. Example 0.002 and )

        CRITICAL RULES:
        1. TRUST THE AUTOMATED GRADE. It has already checked the numbers with a 5% tolerance.
        2. IGNORE ROUNDING: If the grade is "PASS", do not nitpick 1.99 vs 2.0. That is correct.
        3. CHECK SIGNS: If the grade is "FAIL" but the number looks right (e.g. 5 vs -5), it is a SIGN CONVENTION ERROR.

        --- EXAMPLE SCENARIO ---
        Problem: Calculate i1.
        Ground Truth: i1 = -2.0 A
        Student Answer: "The current is 2.0 A."
        Automated Grade: ERR_VAL (or ERR_SIGN)
        -> YOUR DIAGNOSIS: "ERR_SIGN" (Magnitude correct, direction ignored).
        ------------------------

        YOUR TASK:
        Analyze the reasoning chain. Why did they get that grade?

        CLASSIFY THE ERROR (Select ONE):
        - "CORRECT": Math is PASS. Reasoning is sound.
        - "ERR_SIGN_CONVENTION": Math is ERR_SIGN. They ignored the 'Clockwise' or 'Passive Sign' convention.
        - "ERR_METHOD_VIOLATION": Used KCL when asked for KVL (or vice versa).
        - "ERR_PHYSICS_SETUP": Wrote the wrong KVL/KCL equations (fundamental physics error).
        - "ERR_CALCULATION": Equations were correct, but they failed the algebra/arithmetic.
        - "ERR_HALLUCINATION": Invented numbers or stopped working.

        RETURN JSON: {"category": "ERR_...", "explanation": "..."}
        """

        user_prompt = f"""
        --- PROBLEM ---
        {entry['Prompt']}

        --- GROUND TRUTH ---
        {json.dumps(entry['GroundTruth'])}

        --- AUTOMATED MATH GRADE ---
        Verdict: {math_grade}
        (Trust this verdict. If FAIL, find the logic error.)

        --- STUDENT RESPONSE ---
        {entry['Response'][:25000]}
        """

        try:
            # Reasoning models do not support temperature
            response = await client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}
                ]
            )
            content = response.choices[0].message.content

            # Extract JSON from markdown
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0].strip()

            return {**entry, "Diagnosis": json.loads(content)}
        except Exception as e:
            print(f"Error on {entry['ID']}: {e}")
            return {**entry, "Diagnosis": {"category": "API_ERROR"}}

# ==============================================================================
# RUNNER
# ==============================================================================
async def main():
    if not os.path.exists(INPUT_FILE):
        print("Run Step 2 script first!")
        return

    with open(INPUT_FILE, 'r') as f:
        data = json.load(f)

    print(f"Judging {len(data)} entries with {JUDGE_MODEL}...")

    # Conservative limit for GPT-5
    sem = asyncio.Semaphore(5)

    tasks = [judge_entry(row, sem) for row in data]
    results = await asyncio.gather(*tasks)

    with open(OUTPUT_FILE, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✅ Final Diagnosis Saved: {OUTPUT_FILE}")

if __name__ == "__main__":
    asyncio.run(main())

Judging 500 entries with gpt-5...
✅ Final Diagnosis Saved: final_diagnosis_gpt5.json


Judge auditing with sonnet 4.5

In [6]:
import os, json, random, time, re
from typing import Dict, Any, Optional, List, Tuple

from anthropic import Anthropic
from sklearn.metrics import cohen_kappa_score

# =========================
# CONFIG
# =========================
IN_JUDGED_FILE = "final_diagnosis_gpt5.json"  # your GPT-5-judged full dataset
SAMPLE_SIZE = 50
SEED = 999

# Use the strongest Sonnet you have access to.
# If this errors, replace with the model id shown in your Anthropic dashboard.
SONNET_MODEL = "claude-sonnet-4-5-20250929"

OUT_SAMPLE = "audit_sample_50.jsonl"
OUT_SONNET = "sonnet_second_judge_outputs.jsonl"
OUT_REPORT = "audit_report.json"

ALLOWED = [
    "ERR_SIGN_CONVENTION",
    "ERR_METHOD_VIOLATION",
    "ERR_PHYSICS_SETUP",
    "ERR_CALCULATION",
    "ERR_HALLUCINATION",
]

SYSTEM_PROMPT = """You are a strict audit judge for circuit-analysis solutions.
Do NOT solve the circuit. Do NOT introduce new numbers.
Your task is to classify the PRIMARY failure cause using the given rubric and the provided ground truth.
Return ONLY JSON.
"""

RUBRIC = f"""
Choose exactly ONE category from:
{", ".join(ALLOWED)}

Definitions:
- ERR_SIGN_CONVENTION: Primary issue is violating sign/direction conventions (mesh direction, PSC, reference polarity). This often corresponds to sign flips, but may also occur when multiple outputs are near-correct except sign.
- ERR_METHOD_VIOLATION: Used the wrong method (e.g., KCL when KVL requested, or vice versa).
- ERR_PHYSICS_SETUP: Fundamental modeling error (wrong KVL/KCL equations, wrong topology interpretation).
- ERR_CALCULATION: Setup largely correct, but algebra/arithmetic solving is wrong.
- ERR_HALLUCINATION: Invented values/components/variables, or response is incomplete/unusable.

Return ONLY this JSON schema:
{{"category":"<ONE>", "confidence":<0..1>, "reason":"1-2 sentences"}}
"""

def extract_first_json(text: str) -> Optional[Dict[str, Any]]:
    text = text.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.IGNORECASE).strip()
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                chunk = text[start:i+1]
                try:
                    return json.loads(chunk)
                except:
                    return None
    return None

def sample_failures(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    failures = []
    for r in rows:
        diag = (r.get("Diagnosis") or {})
        cat = diag.get("category")
        if cat and cat != "CORRECT" and (r.get("Response") or "").strip():
            failures.append(r)

    if len(failures) < SAMPLE_SIZE:
        raise ValueError(f"Not enough failures to sample {SAMPLE_SIZE}. Found {len(failures)}.")

    random.seed(SEED)
    return random.sample(failures, SAMPLE_SIZE)

def build_prompt(item: Dict[str, Any]) -> str:
    return f"""{RUBRIC}

--- PROBLEM ---
{item.get("Prompt","")}

--- REQUIRED METHOD / CONTEXT ---
Method: {item.get("Method","")}, Type: {item.get("Type","")}

--- GROUND TRUTH (reference only; do not recompute) ---
{json.dumps(item.get("GroundTruth", {}))}

--- STUDENT RESPONSE ---
{(item.get("Response","") or "")[:20000]}
"""

def sonnet_label(client: Anthropic, item: Dict[str, Any], retries=6) -> Dict[str, Any]:
    for attempt in range(1, retries+1):
        try:
            msg = client.messages.create(
                model=SONNET_MODEL,
                max_tokens=1000,
                temperature=0.0,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": build_prompt(item)}],
            )
            text = "".join([blk.text for blk in msg.content if hasattr(blk, "text")]).strip()
            parsed = extract_first_json(text)
            if not parsed:
                raise ValueError("Could not parse JSON output.")
            cat = (parsed.get("category") or "").strip()
            if cat not in ALLOWED:
                raise ValueError(f"Invalid category: {cat}")
            return {
                "category": cat,
                "confidence": parsed.get("confidence", None),
                "reason": parsed.get("reason", ""),
                "raw": text
            }
        except Exception as e:
            if attempt == retries:
                return {"category": "API_ERROR", "confidence": None, "reason": str(e), "raw": None}
            time.sleep(1.25 * attempt)

def main():
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise RuntimeError("Set ANTHROPIC_API_KEY in your environment.")
    client = Anthropic(api_key=api_key)

    with open(IN_JUDGED_FILE, "r") as f:
        rows = json.load(f)

    sample = sample_failures(rows)

    # Save the sample for reproducibility (IDs + GPT5 labels)
    with open(OUT_SAMPLE, "w") as f:
        for r in sample:
            f.write(json.dumps({
                "ID": r.get("ID"),
                "ModelUnderTest": r.get("Model"),
                "Method": r.get("Method"),
                "Type": r.get("Type"),
                "GPT5_category": (r.get("Diagnosis") or {}).get("category")
            }) + "\n")

    outputs = []
    y_true, y_pred = [], []
    for i, r in enumerate(sample, 1):
        time.sleep(2)
        gpt5_cat = (r.get("Diagnosis") or {}).get("category")
        sonnet = sonnet_label(client, r)
        outputs.append({
            "ID": r.get("ID"),
            "ModelUnderTest": r.get("Model"),
            "GPT5_category": gpt5_cat,
            "Sonnet_category": sonnet["category"],
            "Sonnet_confidence": sonnet["confidence"],
            "Sonnet_reason": sonnet["reason"],
            "Sonnet_raw": sonnet.get("raw"),
        })
        if sonnet["category"] in ALLOWED and gpt5_cat in ALLOWED:
            y_true.append(gpt5_cat)
            y_pred.append(sonnet["category"])
        print(f"[{i:02d}/{SAMPLE_SIZE}] {r.get('ID')}  GPT5={gpt5_cat}  Sonnet={sonnet['category']}")

    with open(OUT_SONNET, "w") as f:
        for o in outputs:
            f.write(json.dumps(o) + "\n")

    # Agreement stats (failures only, exclude API_ERROR)
    agreement = sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true) if y_true else 0.0
    kappa = cohen_kappa_score(y_true, y_pred, labels=ALLOWED) if y_true else 0.0

    report = {
        "seed": SEED,
        "sample_size": SAMPLE_SIZE,
        "n_valid_pairs": len(y_true),
        "percent_agreement": agreement,
        "cohens_kappa": kappa,
        "model_second_judge": SONNET_MODEL,
        "note": "Computed on failed subtasks only; CORRECT cases are not sampled.",
    }
    with open(OUT_REPORT, "w") as f:
        json.dump(report, f, indent=2)

    print("\n=== AUDIT REPORT ===")
    print(json.dumps(report, indent=2))
    print(f"\nWrote: {OUT_SAMPLE}, {OUT_SONNET}, {OUT_REPORT}")

if __name__ == "__main__":
    main()


[01/50] Prob3_Bridge_Control-Base  GPT5=ERR_PHYSICS_SETUP  Sonnet=ERR_PHYSICS_SETUP
[02/50] Prob3_Bridge_Trap-Gen-4  GPT5=ERR_SIGN_CONVENTION  Sonnet=ERR_SIGN_CONVENTION
[03/50] Prob5_Dependent_Control-Gen-4  GPT5=ERR_SIGN_CONVENTION  Sonnet=ERR_SIGN_CONVENTION
[04/50] Prob2_Opposing_Control-Gen-3  GPT5=ERR_SIGN_CONVENTION  Sonnet=ERR_SIGN_CONVENTION
[05/50] Prob3_Bridge_Control-Gen-1  GPT5=ERR_PHYSICS_SETUP  Sonnet=ERR_PHYSICS_SETUP
[06/50] Prob2_Opposing_Trap-Gen-4  GPT5=ERR_SIGN_CONVENTION  Sonnet=ERR_SIGN_CONVENTION
[07/50] Prob4_Ladder_Control-Gen-3  GPT5=ERR_HALLUCINATION  Sonnet=ERR_CALCULATION
[08/50] Prob4_Ladder_Control-Gen-2  GPT5=ERR_CALCULATION  Sonnet=ERR_CALCULATION
[09/50] Prob3_Bridge_Control-Gen-4  GPT5=ERR_PHYSICS_SETUP  Sonnet=ERR_PHYSICS_SETUP
[10/50] Prob5_Dependent_Trap-Gen-4  GPT5=ERR_PHYSICS_SETUP  Sonnet=ERR_PHYSICS_SETUP
[11/50] Prob4_Ladder_Trap-Gen-4  GPT5=ERR_PHYSICS_SETUP  Sonnet=ERR_PHYSICS_SETUP
[12/50] Prob2_Opposing_Trap-Gen-4  GPT5=ERR_PHYSICS_SETUP 

The failed checks in the previous script will be run here

In [7]:
import json
from sklearn.metrics import cohen_kappa_score

PATCHED_FILE = "sonnet_second_judge_outputs.jsonl"

ALLOWED_5 = [
    "ERR_SIGN_CONVENTION",
    "ERR_METHOD_VIOLATION",
    "ERR_PHYSICS_SETUP",
    "ERR_CALCULATION",
    "ERR_HALLUCINATION",
]

def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows = load_jsonl(PATCHED_FILE)

# Keep only valid judge pairs (exclude API_ERROR / missing)
pairs_5 = [
    (r.get("GPT5_category"), r.get("Sonnet_category"))
    for r in rows
    if r.get("GPT5_category") in ALLOWED_5 and r.get("Sonnet_category") in ALLOWED_5
]

y_true = [a for a, b in pairs_5]
y_pred = [b for a, b in pairs_5]

n_total = len(rows)
n_valid = len(pairs_5)

agreement_5 = sum(1 for a, b in pairs_5 if a == b) / n_valid if n_valid else 0.0
kappa_5 = cohen_kappa_score(y_true, y_pred, labels=ALLOWED_5) if n_valid else 0.0

print("=== 5-WAY AGREEMENT (fine taxonomy) ===")
print(f"Total rows in file: {n_total}")
print(f"Valid pairs (non-API_ERROR): {n_valid}")
print(f"Percent agreement: {agreement_5:.3%}")
print(f"Cohen's kappa:     {kappa_5:.3f}")

# ---- Optional: 2-way mapping (Compliance vs Competence) ----
COMPLIANCE = {"ERR_SIGN_CONVENTION", "ERR_METHOD_VIOLATION"}
COMPETENCE = {"ERR_PHYSICS_SETUP", "ERR_CALCULATION", "ERR_HALLUCINATION"}

def to_2way(cat: str):
    if cat in COMPLIANCE: return "COMPLIANCE"
    if cat in COMPETENCE: return "COMPETENCE"
    return None

pairs_2 = []
for a, b in pairs_5:
    a2, b2 = to_2way(a), to_2way(b)
    if a2 and b2:
        pairs_2.append((a2, b2))

y2_true = [a for a, b in pairs_2]
y2_pred = [b for a, b in pairs_2]

agreement_2 = sum(1 for a, b in pairs_2 if a == b) / len(pairs_2) if pairs_2 else 0.0
kappa_2 = cohen_kappa_score(y2_true, y2_pred, labels=["COMPLIANCE", "COMPETENCE"]) if pairs_2 else 0.0

print("\n=== 2-WAY AGREEMENT (main paper dichotomy) ===")
print(f"Valid pairs (mapped): {len(pairs_2)}")
print(f"Percent agreement:    {agreement_2:.3%}")
print(f"Cohen's kappa:        {kappa_2:.3f}")


=== 5-WAY AGREEMENT (fine taxonomy) ===
Total rows in file: 50
Valid pairs (non-API_ERROR): 50
Percent agreement: 72.000%
Cohen's kappa:     0.534

=== 2-WAY AGREEMENT (main paper dichotomy) ===
Valid pairs (mapped): 50
Percent agreement:    80.000%
Cohen's kappa:        0.567


In [8]:
import json, pandas as pd, re, math
from statsmodels.stats.proportion import proportion_confint

IN_FILE = "final_diagnosis_gpt5.json"  # your GPT-5 judged dataset

with open(IN_FILE, "r") as f:
    rows = json.load(f)

df = pd.DataFrame(rows)
df["diag_cat"] = df["Diagnosis"].apply(lambda x: x.get("category") if isinstance(x, dict) else x)
df["correct"] = (df["diag_cat"] == "CORRECT")
df["topology"] = df["ID"].str.extract(r"(Prob\d+)")

# --- 95% Wilson CI for overall accuracy per model (N=100 each) ---
summary = []
for model, g in df.groupby("Model"):
    n = len(g)
    k = int(g["correct"].sum())
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    summary.append({
        "Model": model,
        "Acc%": 100*k/n,
        "CI95_low%": 100*lo,
        "CI95_high%": 100*hi,
        "N": n
    })
acc_ci = pd.DataFrame(summary).sort_values("Acc%", ascending=False)
print("=== Overall accuracy with 95% CI ===")
print(acc_ci.to_string(index=False))

# --- Topology-wise accuracy table (N=20/topology/model) ---
topo = (df.groupby(["Model","topology"])["correct"].mean()*100).unstack()
print("\n=== Accuracy by topology (%) ===")
print(topo.round(1).to_string())


=== Overall accuracy with 95% CI ===
          Model  Acc%  CI95_low%  CI95_high%   N
          gpt-5  66.0  56.277729   74.538479 100
Claude Opus 4.5  65.0  55.254443   73.635752 100
        o4-mini  57.0  47.215390   66.266701 100
         gpt-4o  10.0   5.522914   17.436566 100
    GPT-4o Mini   3.0   1.025452    8.451936 100

=== Accuracy by topology (%) ===
topology         Prob1  Prob2  Prob3  Prob4  Prob5
Model                                             
Claude Opus 4.5   60.0   50.0   45.0   95.0   75.0
GPT-4o Mini        5.0    0.0    0.0   10.0    0.0
gpt-4o             0.0    0.0    0.0   20.0   30.0
gpt-5             85.0   30.0   50.0  100.0   65.0
o4-mini           60.0   20.0   65.0   80.0   60.0
